In [ ]:
# =============================================================================
# PALM LULC-Water Quality Correlation Analysis
# =============================================================================
#
# Author: Juan Gabriel Loaiza
# Affiliation: Universidad Autónoma de Sinaloa (UAS), Mexico
# Project: Land-use/land-cover and water-quality analysis of the
#          Adolfo Lopez Mateos Reservoir (PALM), Sinaloa, Mexico
# Repository version: v1.0.0
#
# Description:
# Integrated analysis of LULC change and water-quality change, including
# Pearson, Spearman and period-adjusted partial correlations, Benjamini-
# Hochberg correction, exact and one-year-lagged temporal scenarios, and
# associated reproducibility figures/tables.
#
# License: See repository LICENSE file
# =============================================================================

# ============================================================
# CORRELACIÓN ENTRE CAMBIO DE USO DEL SUELO  ------
# Y CAMBIO EN LA CALIDAD DEL AGUA
# ============================================================
#
# Unidad de análisis:
# Zona de escurrimiento–punto × transición temporal
#
# Análisis principal:
# Uso del suelo 2013–2015 ↔ Calidad del agua 2013–2015
# Uso del suelo 2015–2017 ↔ Calidad del agua 2015–2017
#
# Análisis complementario retardado:
# Uso del suelo 2013–2015 ↔ Calidad del agua 2014–2016
# Uso del suelo 2015–2017 ↔ Calidad del agua 2016–2018
#
# Resultados:
# - Tabla analítica
# - Pearson
# - Spearman
# - Partial correlation controlling for period
# - Regresión ajustada por periodo
# - Corrección Benjamini–Hochberg
# - Análisis dejando una zona fuera
# - Matrices gráficas de correlación
# - Archivo Excel con todos los resultados
# ============================================================


# ============================================================
# 1. INSTALACIÓN DE PAQUETES
# ============================================================



# ============================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

from pathlib import Path
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

from IPython.display import display


# ============================================================
# 3. CONEXIÓN CON GOOGLE DRIVE
# ============================================================



# ============================================================
# 4. CARPETAS DE ENTRADA Y SALIDA
# ============================================================

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "derived_data").exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((q for q in candidates if (q / "derived_data").exists()), Path.cwd())

WQ_INPUT = REPO_ROOT / "derived_data" / "water_quality_predictions"
LULC_INPUT = REPO_ROOT / "derived_data" / "lulc_change"
SALIDA = REPO_ROOT / "results" / "correlation"
SALIDA.mkdir(parents=True, exist_ok=True)
BASE = WQ_INPUT
SALIDA.mkdir(parents=True, exist_ok=True)

print("Input folder:")
print(BASE)

print("\nResults folder:")
print(SALIDA)


# ============================================================
# 5. NOMBRES DE LAS CLASES
# ============================================================
#
# Cuando conozcas la equivalencia real, cambia solamente los
# nombres del lado derecho.
#
# Ejemplo:
# 1: "Agricultura"
# 2: "Forest"
# ============================================================

NOMBRES_CLASES = {
    1: "Water",
    2: "Forest",
    3: "Agricultural land",
    4: "Tropical forests",
    5: "Bare land",
}


# ============================================================
# 6. ARCHIVOS DE CALIDAD DEL AGUA
# ============================================================

ARCHIVOS_CALIDAD = {
    "COT": [
        "Predictions_new_points_TOC_2013-2014.xlsx",
        "Predictions_new_points_TOC_2015-2016.xlsx",
        "Predictions_new_points_TOC_2017-2018.xlsx",
    ],

    "SDT": [
        "Predictions_new_points_TDS_2013-2014.xlsx",
        "Predictions_new_points_TDS_2015-2016.xlsx",
        "Predictions_new_points_TDS_2017-2018.xlsx",
    ],

    "Cl_a": [
        "Predictions_new_points_Cha-a_2013-2014.xlsx",
        "Predictions_new_points_Cha-a_2015-2016.xlsx",
        "Predictions_new_points_Cha-a_2017-2018.xlsx",
    ],
}


# ============================================================
# 7. ARCHIVOS DE CAMBIO DE USO DEL SUELO
# ============================================================

ARCHIVOS_SUELO = {
    (2013, 2015, 1):
        "land_use_change_analysis_2013_2015_zone_1.xlsx",

    (2013, 2015, 2):
        "land_use_change_analysis_2013_2015_zone_2.xlsx",

    (2013, 2015, 3):
        "land_use_change_analysis_2013_2015_zone_3.xlsx",

    (2013, 2015, 4):
        "land_use_change_analysis_2013_2015_zone_4.xlsx",

    (2013, 2015, 5):
        "land_use_change_analysis_2013_2015_zone_5.xlsx",

    (2013, 2015, 6):
        "land_use_change_analysis_2013_2015_zone_6.xlsx",

    (2015, 2017, 1):
        "land_use_change_analysis_2015_2017_zone_1.xlsx",

    (2015, 2017, 2):
        "land_use_change_analysis_2015_2017_zone_2.xlsx",

    (2015, 2017, 3):
        "land_use_change_analysis_2015_2017_zone_3.xlsx",

    (2015, 2017, 4):
        "land_use_change_analysis_2015_2017_zone_4.xlsx",

    (2015, 2017, 5):
        "land_use_change_analysis_2015_2017_zone_5.xlsx",

    (2015, 2017, 6):
        "land_use_change_analysis_2015_2017_zone_6.xlsx",
}


# ============================================================
# 8. VERIFICACIÓN DE LOS ARCHIVOS
# ============================================================

def verificar_archivos():

    requeridos_calidad = [n for lista in ARCHIVOS_CALIDAD.values() for n in lista]
    requeridos_suelo = list(ARCHIVOS_SUELO.values())
    requeridos = requeridos_calidad + requeridos_suelo

    faltantes = (
        [n for n in requeridos_calidad if not (WQ_INPUT / n).exists()] +
        [n for n in requeridos_suelo if not (LULC_INPUT / n).exists()]
    )

    print("\nExpected files:", len(requeridos))
    print("Files found:", len(requeridos) - len(faltantes))

    if faltantes:

        print("\nThe following files were not found:")

        for archivo in faltantes:
            print(" -", archivo)

        raise FileNotFoundError(
            "Faltan uno o más archivos. Revisa la carpeta y los nombres."
        )

    print("\nAll files were found successfully.")


# ============================================================
# 9. LECTURA DE LA CALIDAD DEL AGUA
# ============================================================

def leer_calidad():

    partes = []

    for parametro, archivos in ARCHIVOS_CALIDAD.items():

        for nombre_archivo in archivos:

            ruta = WQ_INPUT / nombre_archivo

            datos = pd.read_excel(
                ruta,
                sheet_name="Predictions"
            )

            columnas_estimadas = [
                columna
                for columna in datos.columns
                if str(columna).endswith("_estimated")
            ]

            if len(columnas_estimadas) != 1:

                raise ValueError(
                    f"No se identificó correctamente la estimación en "
                    f"{nombre_archivo}"
                )

            columna_estimada = columnas_estimadas[0]

            datos = datos[
                ["anio", "punto", columna_estimada]
            ].copy()

            datos = datos.rename(
                columns={columna_estimada: "estimacion"}
            )

            datos["parametro"] = parametro
            datos["archivo_origen"] = nombre_archivo

            # Identifica las dos observaciones de cada año y punto.
            datos["observacion"] = (
                datos.groupby(["anio", "punto"])
                .cumcount() + 1
            )

            partes.append(datos)

    calidad_cruda = pd.concat(
        partes,
        ignore_index=True
    )

    if calidad_cruda["estimacion"].isna().any():

        warnings.warn(
            "Se encontraron estimaciones faltantes. "
            "Solo se omitirán en el cálculo afectado."
        )

    if (calidad_cruda["estimacion"].dropna() <= 0).any():

        raise ValueError(
            "Se encontraron valores iguales o menores que cero. "
            "El cambio logarítmico no puede calcularse."
        )

    # Media geométrica anual por parámetro y punto.
    calidad_anual = (
        calidad_cruda
        .groupby(
            ["parametro", "anio", "punto"],
            as_index=False
        )
        .agg(
            numero_estimaciones=("estimacion", "count"),

            media_geometrica=(
                "estimacion",
                lambda x: np.exp(np.log(x.dropna()).mean())
            ),

            media_aritmetica=("estimacion", "mean"),
            mediana=("estimacion", "median"),
            minimo=("estimacion", "min"),
            maximo=("estimacion", "max"),
            desviacion_estandar=("estimacion", "std")
        )
    )

    return calidad_cruda, calidad_anual


# ============================================================
# 10. CÁLCULO DEL CAMBIO DE CALIDAD DEL AGUA
# ============================================================

def calcular_cambios_calidad(
    calidad_anual,
    pares_anios,
    escenario
):

    filas = []

    for anio_inicial, anio_final in pares_anios:

        inicial = (
            calidad_anual[
                calidad_anual["anio"] == anio_inicial
            ]
            .set_index(["parametro", "punto"])
        )

        final = (
            calidad_anual[
                calidad_anual["anio"] == anio_final
            ]
            .set_index(["parametro", "punto"])
        )

        indices_comunes = inicial.index.intersection(
            final.index
        )

        for parametro, punto in indices_comunes:

            valor_inicial = inicial.loc[
                (parametro, punto),
                "media_geometrica"
            ]

            valor_final = final.loc[
                (parametro, punto),
                "media_geometrica"
            ]

            cambio_logaritmico = (
                100 *
                np.log(valor_final / valor_inicial)
            )

            cambio_porcentual = (
                100 *
                (valor_final - valor_inicial) /
                valor_inicial
            )

            filas.append({
                "zona": int(punto),
                "punto": int(punto),
                "periodo_calidad":
                    f"{anio_inicial}-{anio_final}",
                "anio_inicial_calidad": anio_inicial,
                "anio_final_calidad": anio_final,
                "parametro": parametro,
                "valor_inicial": valor_inicial,
                "valor_final": valor_final,
                "cambio_log": cambio_logaritmico,
                "cambio_porcentual": cambio_porcentual,
                "escenario": escenario
            })

    return pd.DataFrame(filas)


# ============================================================
# 11. LECTURA DEL CAMBIO DE USO DEL SUELO
# ============================================================

def leer_cambios_suelo():

    filas = []

    for clave, nombre_archivo in ARCHIVOS_SUELO.items():

        anio_inicial, anio_final, zona = clave

        ruta = LULC_INPUT / nombre_archivo

        datos = pd.read_excel(
            ruta,
            sheet_name="Change_summary"
        )

        area_total_inicial = datos[
            "Initial Area (ha)"
        ].sum()

        area_total_final = datos[
            "Final Area (ha)"
        ].sum()

        diferencia_area = (
            area_total_final -
            area_total_inicial
        )

        if not np.isclose(
            area_total_inicial,
            area_total_final,
            atol=1.0
        ):

            warnings.warn(
                f"La superficie de la zona {zona}, "
                f"{anio_inicial}-{anio_final}, "
                f"cambió en {diferencia_area:.2f} ha."
            )

        for _, fila in datos.iterrows():

            clase = int(fila["Class"])

            cambio_neto = fila["Net Change (ha)"]
            ganancia = fila["Gain (ha)"]
            perdida = fila["Loss (ha)"]
            cambio_total = fila["Total Change (ha)"]

            # Cambio neto de la clase en puntos porcentuales.
            delta_pp = (
                100 *
                cambio_neto /
                area_total_inicial
            )

            # Ganancia proporcional.
            ganancia_pp = (
                100 *
                ganancia /
                area_total_inicial
            )

            # Pérdida proporcional.
            perdida_pp = (
                100 *
                perdida /
                area_total_inicial
            )

            # Intensidad total de transformación.
            recambio_pp = (
                100 *
                cambio_total /
                area_total_inicial
            )

            filas.append({
                "zona": zona,
                "periodo_suelo":
                    f"{anio_inicial}-{anio_final}",
                "anio_inicial_suelo": anio_inicial,
                "anio_final_suelo": anio_final,
                "clase": clase,
                "nombre_clase": NOMBRES_CLASES[clase],
                "area_total_zona_ha": area_total_inicial,
                "area_inicial_clase_ha":
                    fila["Initial Area (ha)"],
                "area_final_clase_ha":
                    fila["Final Area (ha)"],
                "persistencia_ha":
                    fila["Persistence (ha)"],
                "perdida_ha": perdida,
                "ganancia_ha": ganancia,
                "cambio_neto_ha": cambio_neto,
                "cambio_total_ha": cambio_total,
                "delta_pp": delta_pp,
                "ganancia_pp": ganancia_pp,
                "perdida_pp": perdida_pp,
                "recambio_pp": recambio_pp,
                "archivo_origen": nombre_archivo
            })

    cambios_suelo = pd.DataFrame(filas)

    # Verifica que los cambios de las cinco clases sumen cero.
    comprobacion = (
        cambios_suelo
        .groupby(["zona", "periodo_suelo"])["delta_pp"]
        .sum()
        .reset_index(name="suma_delta_pp")
    )

    if (
        comprobacion["suma_delta_pp"].abs() > 0.01
    ).any():

        warnings.warn(
            "En alguna zona los cambios de las clases "
            "no suman aproximadamente cero."
        )

    return cambios_suelo, comprobacion


# ============================================================
# 12. INTERVALO DE CONFIANZA DE PEARSON
# ============================================================

def intervalo_confianza_pearson(
    r,
    n,
    confianza=0.95
):

    if n <= 3 or abs(r) >= 1:

        return np.nan, np.nan

    alfa = 1 - confianza

    z = np.arctanh(r)

    error = 1 / np.sqrt(n - 3)

    valor_critico = stats.norm.ppf(
        1 - alfa / 2
    )

    inferior = np.tanh(
        z - valor_critico * error
    )

    superior = np.tanh(
        z + valor_critico * error
    )

    return inferior, superior


# ============================================================
# 13. CORRELACIÓN PARCIAL CONTROLANDO EL PERIODO
# ============================================================

def correlacion_parcial_periodo(
    datos,
    columna_x,
    columna_y
):

    # Se resta la media correspondiente a cada periodo.
    residuo_x = (
        datos[columna_x] -
        datos.groupby("periodo_suelo")[
            columna_x
        ].transform("mean")
    )

    residuo_y = (
        datos[columna_y] -
        datos.groupby("periodo_suelo")[
            columna_y
        ].transform("mean")
    )

    resultado = stats.pearsonr(
        residuo_x,
        residuo_y
    )

    return resultado.statistic, resultado.pvalue


# ============================================================
# 14. REGRESIÓN AJUSTADA POR PERIODO
# ============================================================

def regresion_ajustada_periodo(datos):

    periodos = sorted(
        datos["periodo_suelo"].unique()
    )

    indicador_periodo = (
        datos["periodo_suelo"] == periodos[-1]
    ).astype(float).to_numpy()

    x = datos["delta_pp"].to_numpy()
    y = datos["cambio_log"].to_numpy()

    matriz_x = np.column_stack([
        np.ones(len(datos)),
        x,
        indicador_periodo
    ])

    coeficientes = np.linalg.lstsq(
        matriz_x,
        y,
        rcond=None
    )[0]

    predicciones = matriz_x @ coeficientes
    residuos = y - predicciones

    numero_parametros = matriz_x.shape[1]
    grados_libertad = len(y) - numero_parametros

    varianza_residual = (
        residuos @ residuos
    ) / grados_libertad

    covarianza = (
        varianza_residual *
        np.linalg.inv(matriz_x.T @ matriz_x)
    )

    error_estandar = np.sqrt(
        np.diag(covarianza)
    )[1]

    beta = coeficientes[1]

    estadistico_t = beta / error_estandar

    valor_p = (
        2 *
        stats.t.sf(
            abs(estadistico_t),
            grados_libertad
        )
    )

    valor_critico = stats.t.ppf(
        0.975,
        grados_libertad
    )

    limite_inferior = (
        beta -
        valor_critico * error_estandar
    )

    limite_superior = (
        beta +
        valor_critico * error_estandar
    )

    return {
        "beta_delta_pp": beta,
        "beta_error_estandar": error_estandar,
        "beta_t": estadistico_t,
        "beta_p": valor_p,
        "beta_ic95_inferior": limite_inferior,
        "beta_ic95_superior": limite_superior
    }


# ============================================================
# 15. CORRECCIÓN BENJAMINI–HOCHBERG
# ============================================================

def benjamini_hochberg(valores_p):

    valores_p = np.asarray(
        valores_p,
        dtype=float
    )

    numero_pruebas = len(valores_p)

    orden = np.argsort(valores_p)

    valores_ordenados = valores_p[orden]

    ajustados_ordenados = (
        valores_ordenados *
        numero_pruebas /
        np.arange(1, numero_pruebas + 1)
    )

    ajustados_ordenados = np.minimum.accumulate(
        ajustados_ordenados[::-1]
    )[::-1]

    ajustados_ordenados = np.minimum(
        ajustados_ordenados,
        1
    )

    ajustados = np.empty(
        numero_pruebas
    )

    ajustados[orden] = ajustados_ordenados

    return ajustados


# ============================================================
# 16. ANÁLISIS DE CORRELACIÓN
# ============================================================

def ejecutar_correlaciones(
    tabla_analitica,
    escenario
):

    resultados = []

    clases = sorted(
        tabla_analitica["nombre_clase"].unique()
    )

    parametros = sorted(
        tabla_analitica["parametro"].unique()
    )

    for clase, parametro in itertools.product(
        clases,
        parametros
    ):

        datos = tabla_analitica[
            (
                tabla_analitica["nombre_clase"] ==
                clase
            )
            &
            (
                tabla_analitica["parametro"] ==
                parametro
            )
        ].copy()

        datos = datos.dropna(
            subset=["delta_pp", "cambio_log"]
        )

        n = len(datos)

        if n < 4:

            continue

        # Pearson.
        resultado_pearson = stats.pearsonr(
            datos["delta_pp"],
            datos["cambio_log"]
        )

        pearson_r = resultado_pearson.statistic
        pearson_p = resultado_pearson.pvalue

        pearson_ic_inferior, pearson_ic_superior = (
            intervalo_confianza_pearson(
                pearson_r,
                n
            )
        )

        # Spearman.
        resultado_spearman = stats.spearmanr(
            datos["delta_pp"],
            datos["cambio_log"]
        )

        spearman_rho = resultado_spearman.statistic
        spearman_p = resultado_spearman.pvalue

        # Correlación controlando el periodo.
        parcial_r, parcial_p = (
            correlacion_parcial_periodo(
                datos,
                "delta_pp",
                "cambio_log"
            )
        )

        # Regresión ajustada por periodo.
        regresion = regresion_ajustada_periodo(
            datos
        )

        # Análisis dejando una zona fuera.
        correlaciones_sin_zona = []

        for zona in sorted(
            datos["zona"].unique()
        ):

            datos_reducidos = datos[
                datos["zona"] != zona
            ]

            if (
                len(datos_reducidos) >= 3
                and
                datos_reducidos["delta_pp"].nunique() > 1
                and
                datos_reducidos["cambio_log"].nunique() > 1
            ):

                r_sin_zona = stats.pearsonr(
                    datos_reducidos["delta_pp"],
                    datos_reducidos["cambio_log"]
                ).statistic

                correlaciones_sin_zona.append(
                    r_sin_zona
                )

        resultados.append({
            "escenario": escenario,
            "clase": clase,
            "parametro": parametro,
            "n": n,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "pearson_ic95_inferior":
                pearson_ic_inferior,
            "pearson_ic95_superior":
                pearson_ic_superior,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "parcial_r_periodo": parcial_r,
            "parcial_p": parcial_p,
            "beta_delta_pp":
                regresion["beta_delta_pp"],
            "beta_error_estandar":
                regresion["beta_error_estandar"],
            "beta_t":
                regresion["beta_t"],
            "beta_p":
                regresion["beta_p"],
            "beta_ic95_inferior":
                regresion["beta_ic95_inferior"],
            "beta_ic95_superior":
                regresion["beta_ic95_superior"],
            "pearson_sin_zona_minimo":
                min(correlaciones_sin_zona)
                if correlaciones_sin_zona
                else np.nan,
            "pearson_sin_zona_maximo":
                max(correlaciones_sin_zona)
                if correlaciones_sin_zona
                else np.nan
        })

    resultados = pd.DataFrame(
        resultados
    )

    # The 15 land-cover × parameter tests are corrected separately
    # for each correlation method and temporal scenario.
    resultados["pearson_p_BH"] = benjamini_hochberg(
        resultados["pearson_p"]
    )
    resultados["spearman_p_BH"] = benjamini_hochberg(
        resultados["spearman_p"]
    )
    resultados["partial_p_BH"] = benjamini_hochberg(
        resultados["parcial_p"]
    )
    resultados["pearson_significant_BH"] = resultados["pearson_p_BH"] < 0.05
    resultados["spearman_significant_BH"] = resultados["spearman_p_BH"] < 0.05
    resultados["partial_significant_BH"] = resultados["partial_p_BH"] < 0.05

    return resultados


# ============================================================


# ============================================================
# 17. IMPROVED CORRELATION MATRICES
# ============================================================

# Consistent English terminology used only for figures and exported summaries.
CLASS_ORDER = [
    "Water",
    "Forest",
    "Agricultural land",
    "Tropical forests",
    "Bare land",
]

PARAMETER_ORDER = ["TOC", "TDS", "Chl-a"]

PARAMETER_LABELS = {
    "COT": "TOC",
    "TOC": "TOC",
    "SDT": "TDS",
    "TDS": "TDS",
    "Cl_a": "Chl-a",
    "Cl-a": "Chl-a",
    "Cha-a": "Chl-a",
    "Chl-a": "Chl-a",
}

SCENARIO_LABELS = {
    "Exact": "Exact",
    "One_year_lagged": "One-year lagged",
    "One-year lagged": "One-year lagged",
}

MAPA_COLOR = "RdBu_r"
ORDEN_CLASES = CLASS_ORDER
ORDEN_PARAMETROS = PARAMETER_ORDER

CARPETA_SALIDA = SALIDA / "Correlation_matrices"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

METODOS = {
    "Pearson": {
        "coeficiente": "pearson_r",
        "p": "pearson_p",
        "p_BH": "pearson_p_BH",
        "simbolo": "r",
    },
    "Spearman": {
        "coeficiente": "spearman_rho",
        "p": "spearman_p",
        "p_BH": "spearman_p_BH",
        "simbolo": "ρ",
    },
    "Partial": {
        "coeficiente": "parcial_r_periodo",
        "p": "parcial_p",
        "p_BH": "partial_p_BH",
        "simbolo": "partial r",
    },
}


def preparar_resultados_para_figuras(resultados):
    """Adds consistent display labels without changing analytical fields."""
    tabla = resultados.copy()
    tabla["clase_nombre"] = tabla["clase"].astype(str).str.strip()
    tabla["parametro_nombre"] = tabla["parametro"].map(PARAMETER_LABELS)
    tabla["escenario_nombre"] = tabla["escenario"].map(SCENARIO_LABELS)

    if tabla["parametro_nombre"].isna().any():
        desconocidos = tabla.loc[
            tabla["parametro_nombre"].isna(), "parametro"
        ].unique()
        raise ValueError(f"Unrecognized parameters: {desconocidos}")

    if tabla["escenario_nombre"].isna().any():
        desconocidos = tabla.loc[
            tabla["escenario_nombre"].isna(), "escenario"
        ].unique()
        raise ValueError(f"Unrecognized scenarios: {desconocidos}")

    return tabla




def crear_matriz(
    datos,
    columna_valor
):

    matriz = datos.pivot(
        index="clase_nombre",
        columns="parametro_nombre",
        values=columna_valor
    )

    matriz = matriz.reindex(
        index=ORDEN_CLASES,
        columns=ORDEN_PARAMETROS
    )

    return matriz


# ============================================================
# 14. ETIQUETAS DE LAS CELDAS
# ============================================================

def crear_anotaciones(
    matriz_coeficiente,
    matriz_p_BH
):

    anotaciones = pd.DataFrame(
        "",
        index=matriz_coeficiente.index,
        columns=matriz_coeficiente.columns
    )

    for clase in matriz_coeficiente.index:

        for parametro in matriz_coeficiente.columns:

            coeficiente = (
                matriz_coeficiente.loc[
                    clase,
                    parametro
                ]
            )

            p_ajustado = (
                matriz_p_BH.loc[
                    clase,
                    parametro
                ]
            )

            if pd.isna(coeficiente):

                texto = ""

            else:

                # El asterisco se agrega exclusivamente
                # cuando p ajustado por BH es menor a 0.05.
                asterisco = (
                    "*"
                    if p_ajustado < 0.05
                    else ""
                )

                texto = (
                    f"{coeficiente:.2f}"
                    f"{asterisco}"
                )

            anotaciones.loc[
                clase,
                parametro
            ] = texto

    return anotaciones


# ============================================================
# 15. FUNCIÓN PARA UNA MATRIZ INDIVIDUAL
# ============================================================

def dibujar_matriz(
    eje,
    datos_escenario,
    nombre_metodo,
    mostrar_eje_y=True,
    mostrar_barra=False,
    titulo=None
):

    configuracion = (
        METODOS[nombre_metodo]
    )

    matriz_coeficiente = crear_matriz(
        datos_escenario,
        configuracion["coeficiente"]
    )

    matriz_p_BH = crear_matriz(
        datos_escenario,
        configuracion["p_BH"]
    )

    anotaciones = crear_anotaciones(
        matriz_coeficiente,
        matriz_p_BH
    )

    sns.heatmap(
        matriz_coeficiente,
        ax=eje,
        cmap=MAPA_COLOR,
        vmin=-1,
        vmax=1,
        center=0,
        annot=anotaciones,
        fmt="",
        annot_kws={
            "fontsize": 12,
            "fontweight": "normal"
        },
        linewidths=0.7,
        linecolor="white",
        square=False,
        cbar=mostrar_barra,
        cbar_kws={
            "label": "Correlation coefficient",
            "shrink": 0.85
        }
    )

    eje.set_title(
        titulo if titulo else nombre_metodo,
        fontweight="normal",
        pad=10
    )

    eje.set_xlabel(
        "Water quality parameter",
        fontweight="normal"
    )

    if mostrar_eje_y:

        eje.set_ylabel(
            "Land cover",
            fontweight="normal"
        )

        eje.set_yticklabels(
            eje.get_yticklabels(),
            rotation=0
        )

    else:

        eje.set_ylabel("")

        eje.set_yticklabels([])

        eje.tick_params(
            axis="y",
            left=False
        )

    eje.set_xticklabels(
        eje.get_xticklabels(),
        rotation=0
    )

    return (
        matriz_coeficiente,
        matriz_p_BH,
        anotaciones
    )


# ============================================================
# 16. FIGURA COMBINADA POR ESCENARIO
# ============================================================

def crear_figura_combinada(
    resultados,
    escenario
):

    datos_escenario = resultados[
        resultados["escenario_nombre"]
        == escenario
    ].copy()

    if datos_escenario.empty:

        raise ValueError(
            f"No existen datos para {escenario}."
        )

    # Se reserva una columna exclusiva para la barra de color.
    # Así, la barra no invade ni comprime la tercera matriz.
    figura = plt.figure(
        figsize=(17.5, 6.2),
        constrained_layout=False
    )

    rejilla = figura.add_gridspec(
        nrows=1,
        ncols=4,
        width_ratios=[1, 1, 1, 0.055],
        left=0.14,
        right=0.96,
        top=0.84,
        bottom=0.18,
        wspace=0.20
    )

    ejes = np.array([
        figura.add_subplot(rejilla[0, 0]),
        figura.add_subplot(rejilla[0, 1]),
        figura.add_subplot(rejilla[0, 2])
    ])

    eje_barra = figura.add_subplot(
        rejilla[0, 3]
    )

    matrices = {}

    titulos = {
        "Pearson":
            "Pearson correlation",

        "Spearman":
            "Spearman correlation",

        "Partial":
            "Partial correlation by period"
    }

    for indice, metodo in enumerate(
        [
            "Pearson",
            "Spearman",
            "Partial"
        ]
    ):

        matriz, matriz_p_BH, anotaciones = (
            dibujar_matriz(
                eje=ejes[indice],
                datos_escenario=datos_escenario,
                nombre_metodo=metodo,
                mostrar_eje_y=(indice == 0),
                mostrar_barra=False,
                titulo=titulos[metodo]
            )
        )

        matrices[metodo] = {
            "coeficientes": matriz,
            "p_BH": matriz_p_BH,
            "anotaciones": anotaciones
        }

    # Barra de colores común.
    normalizacion = Normalize(
        vmin=-1,
        vmax=1
    )

    objeto_color = ScalarMappable(
        norm=normalizacion,
        cmap=MAPA_COLOR
    )

    objeto_color.set_array([])

    barra = figura.colorbar(
        objeto_color,
        cax=eje_barra,
        orientation="vertical"
    )

    barra.set_label(
        "Correlation coefficient",
        fontweight="normal"
    )

    titulo_escenario = (
        "Exact temporal scenario"
        if escenario == "Exact"
        else "Scenario with a one-year lagged response"
    )

    figura.suptitle(
        titulo_escenario,
        fontsize=16,
        fontweight="normal",
        y=0.99
    )

    figura.text(
        0.5,
        0.025,
        (
            "* Significant association after Benjamini–Hochberg correction "
            "(adjusted p < 0.05)"
        ),
        ha="center",
        va="center",
        fontsize=11,
        fontweight="normal",
        color="#444444"
    )

    nombre_seguro = (
        "exact"
        if escenario == "Exact"
        else "one_year_lagged"
    )

    ruta_png = (
        CARPETA_SALIDA
        / f"Correlation_matrices_{nombre_seguro}.png"
    )

    ruta_pdf = (
        CARPETA_SALIDA
        / f"Correlation_matrices_{nombre_seguro}.pdf"
    )

    ruta_svg = (
        CARPETA_SALIDA
        / f"Correlation_matrices_{nombre_seguro}.svg"
    )

    # Obliga al backend de Colab a completar el dibujo antes de guardar.
    figura.canvas.draw()

    figura.savefig(
        ruta_png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none"
    )

    figura.savefig(
        ruta_pdf,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none"
    )

    figura.savefig(
        ruta_svg,
        bbox_inches="tight",
        facecolor="white",
        edgecolor="none"
    )

    plt.show()
    plt.close(figura)

    return (
        matrices,
        ruta_png,
        ruta_pdf,
        ruta_svg
    )


# ============================================================
# 17. MATRICES INDIVIDUALES
# ============================================================

def crear_matrices_individuales(
    resultados,
    escenario
):

    datos_escenario = resultados[
        resultados["escenario_nombre"]
        == escenario
    ].copy()

    rutas = []

    nombre_escenario = (
        "exact"
        if escenario == "Exact"
        else "one_year_lagged"
    )

    titulos = {
        "Pearson":
            "Pearson correlation",

        "Spearman":
            "Spearman correlation",

        "Partial":
            "Partial correlation controlling for period"
    }

    for metodo in [
        "Pearson",
        "Spearman",
        "Partial"
    ]:

        figura, eje = plt.subplots(
            figsize=(7.5, 6)
        )

        dibujar_matriz(
            eje=eje,
            datos_escenario=datos_escenario,
            nombre_metodo=metodo,
            mostrar_eje_y=True,
            mostrar_barra=True,
            titulo=titulos[metodo]
        )

        figura.text(
            0.5,
            0.015,
            (
                "* Benjamini–Hochberg adjusted p < 0.05"
            ),
            ha="center",
            fontsize=10.5,
            fontweight="normal",
            color="#444444"
        )

        figura.tight_layout(
            rect=[0, 0.05, 1, 1]
        )

        nombre_metodo = (
            metodo.lower()
            .replace(" ", "_")
        )

        ruta_png = (
            CARPETA_SALIDA
            / (
                f"Matrix_{nombre_metodo}_"
                f"{nombre_escenario}.png"
            )
        )

        ruta_pdf = (
            CARPETA_SALIDA
            / (
                f"Matrix_{nombre_metodo}_"
                f"{nombre_escenario}.pdf"
            )
        )

        # Renderizar explícitamente antes de exportar evita archivos vacíos
        # con algunos backends interactivos de Google Colab.
        figura.canvas.draw()

        figura.savefig(
            ruta_png,
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none"
        )

        figura.savefig(
            ruta_pdf,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="none"
        )

        plt.show()
        plt.close(figura)

        rutas.extend([
            ruta_png,
            ruta_pdf
        ])

    return rutas


# ============================================================

# English headers used in every exported table.
OUTPUT_COLUMNS_EN = {'anio': 'year', 'punto': 'point', 'zona': 'zone', 'parametro': 'parameter', 'estimacion': 'estimate', 'archivo_origen': 'source_file', 'observacion': 'observation', 'numero_estimaciones': 'number_of_estimates', 'media_geometrica': 'geometric_mean', 'media_aritmetica': 'arithmetic_mean', 'mediana': 'median', 'minimo': 'minimum', 'maximo': 'maximum', 'desviacion_estandar': 'standard_deviation', 'error_inferior': 'lower_error', 'error_superior': 'upper_error', 'periodo_calidad': 'water_quality_period', 'anio_inicial_calidad': 'water_quality_initial_year', 'anio_final_calidad': 'water_quality_final_year', 'valor_inicial': 'initial_value', 'valor_final': 'final_value', 'cambio_log': 'log_change', 'cambio_porcentual': 'percent_change', 'periodo_suelo': 'land_use_period', 'anio_inicial_suelo': 'land_use_initial_year', 'anio_final_suelo': 'land_use_final_year', 'clase': 'class', 'nombre_clase': 'class_name', 'area_total_zona_ha': 'total_zone_area_ha', 'area_inicial_clase_ha': 'initial_class_area_ha', 'area_final_clase_ha': 'final_class_area_ha', 'persistencia_ha': 'persistence_ha', 'perdida_ha': 'loss_ha', 'ganancia_ha': 'gain_ha', 'cambio_neto_ha': 'net_change_ha', 'cambio_total_ha': 'total_change_ha', 'ganancia_pp': 'gain_pp', 'perdida_pp': 'loss_pp', 'recambio_pp': 'turnover_pp', 'escenario': 'scenario', 'parcial_r_periodo': 'partial_r_period', 'parcial_p': 'partial_p', 'parcial_p_ajustado_BH': 'partial_p_adjusted_BH', 'significativa_BH_0_05': 'significant_BH_0_05', 'area_ha': 'area_ha', 'fuente': 'source', 'superficie_total_ha': 'total_area_ha', 'porcentaje': 'percentage', 'diferencia_ha': 'difference_ha', 'diferencia_absoluta_ha': 'absolute_difference_ha', 'escenario_nombre': 'scenario_name', 'clase_nombre': 'class_name', 'parametro_nombre': 'parameter_name', 'parcial_p_BH_nuevo': 'partial_p_BH'}

def for_export(table):
    return table.rename(columns=OUTPUT_COLUMNS_EN)

# ============================================================
# 18. EXECUTION
# ============================================================

verificar_archivos()

print("\nReading water-quality files...")
calidad_cruda, calidad_anual = leer_calidad()
print("Original water-quality records:", len(calidad_cruda))
print("Annual summaries:", len(calidad_anual))

print("\nReading land-use files...")
cambios_suelo, comprobacion_suelo = leer_cambios_suelo()
print("Land-use change records:", len(cambios_suelo))


# Exact temporal alignment.
calidad_exacta = calcular_cambios_calidad(
    calidad_anual,
    [(2013, 2015), (2015, 2017)],
    "Exact",
)
calidad_exacta["periodo_suelo"] = calidad_exacta["periodo_calidad"]

tabla_exacta = cambios_suelo.merge(
    calidad_exacta,
    on=["zona", "periodo_suelo"],
    how="inner",
    validate="many_to_many",
)


# One-year lagged water-quality response.
calidad_retardada = calcular_cambios_calidad(
    calidad_anual,
    [(2014, 2016), (2016, 2018)],
    "One_year_lagged",
)
calidad_retardada["periodo_suelo"] = calidad_retardada["periodo_calidad"].map({
    "2014-2016": "2013-2015",
    "2016-2018": "2015-2017",
})

tabla_retardada = cambios_suelo.merge(
    calidad_retardada,
    on=["zona", "periodo_suelo"],
    how="inner",
    validate="many_to_many",
)


def verificar_tabla_analitica(tabla, nombre):
    expected = 6 * 2 * 5 * 3
    if len(tabla) != expected:
        warnings.warn(
            f"{nombre} contains {len(tabla)} rows; {expected} were expected "
            "(6 zones × 2 periods × 5 classes × 3 parameters)."
        )

    duplicados = tabla.duplicated(
        ["zona", "periodo_suelo", "nombre_clase", "parametro"]
    )
    if duplicados.any():
        raise ValueError(f"{nombre} contains duplicated analytical units.")


verificar_tabla_analitica(tabla_exacta, "Exact analysis table")
verificar_tabla_analitica(tabla_retardada, "Lagged analysis table")

resultados_exactos = ejecutar_correlaciones(tabla_exacta, "Exact")
resultados_retardados = ejecutar_correlaciones(
    tabla_retardada, "One_year_lagged"
)

resultados_correlacion = pd.concat(
    [resultados_exactos, resultados_retardados],
    ignore_index=True,
)

resultados_figuras = preparar_resultados_para_figuras(
    resultados_correlacion
)

print("\nCorrelation results:")
display(resultados_figuras.round(4))


# ============================================================
# 19. MATRIX FIGURES
# ============================================================

print("\nGenerating exact-scenario matrices...")
(
    matrices_exactas,
    ruta_exacta_png,
    ruta_exacta_pdf,
    ruta_exacta_svg,
) = crear_figura_combinada(resultados_figuras, escenario="Exact")

rutas_individuales_exactas = crear_matrices_individuales(
    resultados_figuras,
    escenario="Exact",
)

print("\nGenerating one-year-lagged matrices...")
(
    matrices_retardadas,
    ruta_retardada_png,
    ruta_retardada_pdf,
    ruta_retardada_svg,
) = crear_figura_combinada(
    resultados_figuras,
    escenario="One-year lagged",
)

rutas_individuales_retardadas = crear_matrices_individuales(
    resultados_figuras,
    escenario="One-year lagged",
)


# ============================================================
# 20. ENGLISH EXPORT HELPERS
# ============================================================

OUTPUT_COLUMNS_EN = {
    "anio": "year",
    "punto": "point",
    "zona": "zone",
    "parametro": "parameter",
    "estimacion": "estimate",
    "archivo_origen": "source_file",
    "observacion": "observation",
    "numero_estimaciones": "number_of_estimates",
    "media_geometrica": "geometric_mean",
    "media_aritmetica": "arithmetic_mean",
    "mediana": "median",
    "minimo": "minimum",
    "maximo": "maximum",
    "desviacion_estandar": "standard_deviation",
    "periodo_calidad": "water_quality_period",
    "anio_inicial_calidad": "water_quality_initial_year",
    "anio_final_calidad": "water_quality_final_year",
    "valor_inicial": "initial_value",
    "valor_final": "final_value",
    "cambio_log": "log_change",
    "cambio_porcentual": "percent_change",
    "periodo_suelo": "land_use_period",
    "anio_inicial_suelo": "land_use_initial_year",
    "anio_final_suelo": "land_use_final_year",
    "clase": "class",
    "nombre_clase": "class_name",
    "area_total_zona_ha": "total_zone_area_ha",
    "area_inicial_clase_ha": "initial_class_area_ha",
    "area_final_clase_ha": "final_class_area_ha",
    "persistencia_ha": "persistence_ha",
    "perdida_ha": "loss_ha",
    "ganancia_ha": "gain_ha",
    "cambio_neto_ha": "net_change_ha",
    "cambio_total_ha": "total_change_ha",
    "ganancia_pp": "gain_pp",
    "perdida_pp": "loss_pp",
    "recambio_pp": "turnover_pp",
    "escenario": "scenario",
    "parcial_r_periodo": "partial_r_period",
    "parcial_p": "partial_p",
    "clase_nombre": "class_name_display",
    "parametro_nombre": "parameter_name",
    "escenario_nombre": "scenario_name",
}


def for_export(tabla):
    return tabla.rename(columns=OUTPUT_COLUMNS_EN)


# ============================================================
# 21. SINGLE EXCEL WORKBOOK
# ============================================================

archivo_excel = SALIDA / "Unified_land_use_water_quality_results.xlsx"

with pd.ExcelWriter(archivo_excel, engine="openpyxl") as writer:
    for_export(calidad_cruda).to_excel(
        writer, sheet_name="Raw_water_quality", index=False
    )
    for_export(calidad_anual).to_excel(
        writer, sheet_name="Annual_water_quality", index=False
    )
    for_export(cambios_suelo).to_excel(
        writer, sheet_name="Land_use_changes", index=False
    )
    for_export(comprobacion_suelo).to_excel(
        writer, sheet_name="Land_use_check", index=False
    )
    for_export(tabla_exacta).to_excel(
        writer, sheet_name="Exact_analysis_table", index=False
    )
    for_export(tabla_retardada).to_excel(
        writer, sheet_name="Lagged_analysis_table", index=False
    )
    for_export(resultados_figuras).to_excel(
        writer, sheet_name="Correlation_results", index=False
    )

    for method, content in matrices_exactas.items():
        content["coeficientes"].to_excel(
            writer, sheet_name=f"{method}_exact"[:31]
        )
        content["p_BH"].to_excel(
            writer, sheet_name=f"pBH_{method}_exact"[:31]
        )

    for method, content in matrices_retardadas.items():
        content["coeficientes"].to_excel(
            writer, sheet_name=f"{method}_lagged"[:31]
        )
        content["p_BH"].to_excel(
            writer, sheet_name=f"pBH_{method}_lagged"[:31]
        )


# CSV files useful for reproducibility.
for_export(tabla_exacta).to_csv(
    SALIDA / "Exact_analysis_table.csv", index=False, encoding="utf-8-sig"
)
for_export(tabla_retardada).to_csv(
    SALIDA / "Lagged_analysis_table.csv", index=False, encoding="utf-8-sig"
)
for_export(resultados_figuras).to_csv(
    SALIDA / "Correlation_results.csv", index=False, encoding="utf-8-sig"
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print("\nAnalysis completed successfully.")
print("Excel workbook:")
print(archivo_excel)
print("\nResults folder:")
print(SALIDA)

for method, p_column in {
    "Pearson": "pearson_p_BH",
    "Spearman": "spearman_p_BH",
    "Partial": "partial_p_BH",
}.items():
    significant = resultados_figuras[resultados_figuras[p_column] < 0.05]
    print(
        f"{method}: {len(significant)} significant associations "
        "after Benjamini–Hochberg correction."
    )


In [ ]:
# ============================================================
# SERIES TEMPORALES DE CALIDAD DEL AGUA
# COT, SDT Y Cl-a EN SEIS PUNTOS TRIBUTARIOS
# ============================================================
#
# La línea representa la media geométrica anual de las dos
# estimaciones disponibles para cada punto.
#
# Las líneas verticales representan el intervalo entre la
# estimación mínima y máxima de cada año.
#
# Archivos de entrada:
# [legacy Google Drive path removed]
#
# Archivos de salida:
# [legacy Google Drive path removed]/
# Resultados_correlacion/Figuras_series_temporales
# ============================================================


# ============================================================
# 1. INSTALACIÓN DE PAQUETES
# ============================================================



# ============================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from IPython.display import display


# ============================================================
# 3. CONEXIÓN CON GOOGLE DRIVE
# ============================================================



# ============================================================
# 4. CARPETAS
# ============================================================

CARPETA_DATOS = REPO_ROOT / "derived_data" / "water_quality_predictions"

CARPETA_SALIDA = (
    REPO_ROOT / "results" / "correlation" / "Time_series_figures"
)

CARPETA_SALIDA.mkdir(
    parents=True,
    exist_ok=True
)

print("Data folder:")
print(CARPETA_DATOS)

print("\nOutput folder:")
print(CARPETA_SALIDA)


# ============================================================
# 5. ARCHIVOS DE CALIDAD DEL AGUA
# ============================================================

ARCHIVOS_CALIDAD = {
    "COT": [
        "Predictions_new_points_TOC_2013-2014.xlsx",
        "Predictions_new_points_TOC_2015-2016.xlsx",
        "Predictions_new_points_TOC_2017-2018.xlsx",
    ],

    "SDT": [
        "Predictions_new_points_TDS_2013-2014.xlsx",
        "Predictions_new_points_TDS_2015-2016.xlsx",
        "Predictions_new_points_TDS_2017-2018.xlsx",
    ],

    "Cl-a": [
        "Predictions_new_points_Cha-a_2013-2014.xlsx",
        "Predictions_new_points_Cha-a_2015-2016.xlsx",
        "Predictions_new_points_Cha-a_2017-2018.xlsx",
    ],
}


# ============================================================
# 6. UNIDADES DE LOS PARÁMETROS
# ============================================================

UNIDADES = {
    "COT": "TOC (mg/L)",
    "SDT": "TDS (mg/L)",
    "Cl-a": r"Chl-a (mg/m$^3$)",
}


# ============================================================
# 7. COLORES Y MARCADORES
# ============================================================

COLORES_PUNTOS = {
    1: "#1f77b4",
    2: "#ff7f0e",
    3: "#2ca02c",
    4: "#d62728",
    5: "#9467bd",
    6: "#8c564b",
}

MARCADORES_PUNTOS = {
    1: "o",
    2: "s",
    3: "^",
    4: "D",
    5: "P",
    6: "X",
}


# ============================================================
# 8. CONFIGURACIÓN GENERAL DE LAS GRÁFICAS
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "axes.linewidth": 0.8,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 120,
    "savefig.dpi": 600,
})


# ============================================================
# 9. VERIFICACIÓN DE ARCHIVOS
# ============================================================

def verificar_archivos():

    archivos_esperados = [
        nombre
        for lista_archivos in ARCHIVOS_CALIDAD.values()
        for nombre in lista_archivos
    ]

    archivos_faltantes = [
        nombre
        for nombre in archivos_esperados
        if not (CARPETA_DATOS / nombre).exists()
    ]

    print("\nExpected files:", len(archivos_esperados))

    print(
        "Files found:",
        len(archivos_esperados) - len(archivos_faltantes)
    )

    if archivos_faltantes:

        print("\nThe following files were not found:")

        for nombre in archivos_faltantes:
            print(" -", nombre)

        raise FileNotFoundError(
            "Faltan archivos de calidad del agua."
        )

    print(
        "\nTodos los archivos de calidad "
        "fueron encontrados correctamente."
    )


# ============================================================
# 10. LECTURA DE LOS ARCHIVOS
# ============================================================

def leer_archivos_calidad():

    tablas = []

    for parametro, archivos in ARCHIVOS_CALIDAD.items():

        for nombre_archivo in archivos:

            ruta_archivo = (
                CARPETA_DATOS
                / nombre_archivo
            )

            datos = pd.read_excel(
                ruta_archivo,
                sheet_name="Predictions"
            )

            columnas_estimadas = [
                columna
                for columna in datos.columns
                if str(columna).endswith("_estimated")
            ]

            if len(columnas_estimadas) != 1:

                raise ValueError(
                    "No fue posible identificar la columna "
                    f"estimada en {nombre_archivo}"
                )

            columna_estimada = (
                columnas_estimadas[0]
            )

            tabla = datos[
                [
                    "anio",
                    "punto",
                    columna_estimada
                ]
            ].copy()

            tabla = tabla.rename(
                columns={
                    columna_estimada: "estimacion"
                }
            )

            tabla["parametro"] = parametro

            tabla["archivo_origen"] = (
                nombre_archivo
            )

            tabla["observacion"] = (
                tabla
                .groupby(
                    ["anio", "punto"]
                )
                .cumcount() + 1
            )

            tablas.append(tabla)

    calidad_cruda = pd.concat(
        tablas,
        ignore_index=True
    )

    return calidad_cruda


# ============================================================
# 11. RESUMEN ANUAL
# ============================================================

def calcular_resumen_anual(calidad_cruda):

    if calidad_cruda["estimacion"].isna().any():

        print(
            "\nAdvertencia: existen estimaciones faltantes."
        )

    valores_validos = (
        calidad_cruda["estimacion"]
        .dropna()
    )

    if (valores_validos <= 0).any():

        raise ValueError(
            "La media geométrica requiere valores "
            "mayores que cero."
        )

    resumen = (
        calidad_cruda
        .groupby(
            [
                "parametro",
                "anio",
                "punto"
            ],
            as_index=False
        )
        .agg(
            numero_estimaciones=(
                "estimacion",
                "count"
            ),

            media_geometrica=(
                "estimacion",
                lambda x: np.exp(
                    np.log(x.dropna()).mean()
                )
            ),

            media_aritmetica=(
                "estimacion",
                "mean"
            ),

            mediana=(
                "estimacion",
                "median"
            ),

            minimo=(
                "estimacion",
                "min"
            ),

            maximo=(
                "estimacion",
                "max"
            ),

            desviacion_estandar=(
                "estimacion",
                "std"
            )
        )
    )

    resumen["error_inferior"] = (
        resumen["media_geometrica"]
        - resumen["minimo"]
    )

    resumen["error_superior"] = (
        resumen["maximo"]
        - resumen["media_geometrica"]
    )

    return resumen


# ============================================================
# 12. FUNCIÓN PARA CREAR LA FIGURA COMBINADA
# ============================================================

def crear_figura_combinada(resumen):

    parametros = [
        "COT",
        "SDT",
        "Cl-a"
    ]

    figura, ejes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=(10, 12),
        sharex=True
    )

    for eje, parametro in zip(
        ejes,
        parametros
    ):

        datos_parametro = resumen[
            resumen["parametro"] == parametro
        ].copy()

        for punto in range(1, 7):

            datos_punto = (
                datos_parametro[
                    datos_parametro["punto"] == punto
                ]
                .sort_values("anio")
            )

            errores = np.vstack([
                datos_punto["error_inferior"],
                datos_punto["error_superior"]
            ])

            eje.errorbar(
                datos_punto["anio"],
                datos_punto["media_geometrica"],
                yerr=errores,
                color=COLORES_PUNTOS[punto],
                marker=MARCADORES_PUNTOS[punto],
                markersize=5.5,
                markerfacecolor="white",
                markeredgecolor=COLORES_PUNTOS[punto],
                markeredgewidth=1.2,
                linewidth=1.5,
                elinewidth=0.8,
                capsize=2.5,
                capthick=0.8,
                alpha=0.95,
                label=f"Point {punto}",
                zorder=3
            )

        eje.set_ylabel(
            UNIDADES[parametro],
            fontweight="normal"
        )

        eje.set_title(
            f"Serie temporal de {parametro}",
            fontweight="normal",
            pad=8
        )

        eje.grid(
            axis="both",
            linestyle="--",
            linewidth=0.5,
            alpha=0.35
        )

        eje.spines["top"].set_visible(False)
        eje.spines["right"].set_visible(False)

        eje.tick_params(
            axis="both",
            direction="out"
        )

        eje.margins(
            x=0.03
        )

    ejes[-1].set_xlabel(
        "Year",
        fontweight="normal"
    )

    ejes[-1].set_xticks(
        range(2013, 2019)
    )

    # Leyenda compartida debajo de los paneles.
    manejadores, etiquetas = (
        ejes[0].get_legend_handles_labels()
    )

    figura.legend(
        manejadores,
        etiquetas,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=6,
        frameon=False,
        columnspacing=1.5,
        handletextpad=0.5
    )

    figura.suptitle(
        (
            "Temporal variation in water quality "
            "at tributary points"
        ),
        fontsize=16,
        fontweight="normal",
        y=0.995
    )

    figura.tight_layout(
        rect=[0, 0.06, 1, 0.98],
        h_pad=2.0
    )

    ruta_png = (
        CARPETA_SALIDA
        / "Water_quality_time_series.png"
    )

    ruta_pdf = (
        CARPETA_SALIDA
        / "Water_quality_time_series.pdf"
    )

    ruta_svg = (
        CARPETA_SALIDA
        / "Water_quality_time_series.svg"
    )

    figura.savefig(
        ruta_png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_svg,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    return ruta_png, ruta_pdf, ruta_svg


# ============================================================
# 13. FIGURAS INDIVIDUALES
# ============================================================

def crear_figuras_individuales(resumen):

    parametros = [
        "COT",
        "SDT",
        "Cl-a"
    ]

    rutas = []

    for parametro in parametros:

        datos_parametro = resumen[
            resumen["parametro"] == parametro
        ].copy()

        figura, eje = plt.subplots(
            figsize=(9, 5.5)
        )

        for punto in range(1, 7):

            datos_punto = (
                datos_parametro[
                    datos_parametro["punto"] == punto
                ]
                .sort_values("anio")
            )

            errores = np.vstack([
                datos_punto["error_inferior"],
                datos_punto["error_superior"]
            ])

            eje.errorbar(
                datos_punto["anio"],
                datos_punto["media_geometrica"],
                yerr=errores,
                color=COLORES_PUNTOS[punto],
                marker=MARCADORES_PUNTOS[punto],
                markersize=6,
                markerfacecolor="white",
                markeredgecolor=COLORES_PUNTOS[punto],
                markeredgewidth=1.2,
                linewidth=1.6,
                elinewidth=0.8,
                capsize=2.5,
                capthick=0.8,
                label=f"Point {punto}",
                zorder=3
            )

        eje.set_xlabel(
            "Year",
            fontweight="normal"
        )

        eje.set_ylabel(
            UNIDADES[parametro],
            fontweight="normal"
        )

        eje.set_title(
            (
                f"Temporal variation of {parametro} "
                "at tributary points"
            ),
            fontweight="normal",
            pad=10
        )

        eje.set_xticks(
            range(2013, 2019)
        )

        eje.grid(
            axis="both",
            linestyle="--",
            linewidth=0.5,
            alpha=0.35
        )

        eje.spines["top"].set_visible(False)
        eje.spines["right"].set_visible(False)

        eje.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.14),
            ncol=6,
            frameon=False,
            columnspacing=1.3,
            handletextpad=0.5
        )

        figura.tight_layout()

        nombre_seguro = (
            parametro
            .replace("-", "_")
        )

        ruta_png = (
            CARPETA_SALIDA
            / f"Serie_temporal_{nombre_seguro}.png"
        )

        ruta_pdf = (
            CARPETA_SALIDA
            / f"Serie_temporal_{nombre_seguro}.pdf"
        )

        figura.savefig(
            ruta_png,
            dpi=600,
            bbox_inches="tight",
            facecolor="white"
        )

        figura.savefig(
            ruta_pdf,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()

        rutas.extend([
            ruta_png,
            ruta_pdf
        ])

    return rutas


# ============================================================
# 14. EXPORTACIÓN DE LOS DATOS A EXCEL
# ============================================================


# English headers used in every exported table.
OUTPUT_COLUMNS_EN = {'anio': 'year', 'punto': 'point', 'zona': 'zone', 'parametro': 'parameter', 'estimacion': 'estimate', 'archivo_origen': 'source_file', 'observacion': 'observation', 'numero_estimaciones': 'number_of_estimates', 'media_geometrica': 'geometric_mean', 'media_aritmetica': 'arithmetic_mean', 'mediana': 'median', 'minimo': 'minimum', 'maximo': 'maximum', 'desviacion_estandar': 'standard_deviation', 'error_inferior': 'lower_error', 'error_superior': 'upper_error', 'periodo_calidad': 'water_quality_period', 'anio_inicial_calidad': 'water_quality_initial_year', 'anio_final_calidad': 'water_quality_final_year', 'valor_inicial': 'initial_value', 'valor_final': 'final_value', 'cambio_log': 'log_change', 'cambio_porcentual': 'percent_change', 'periodo_suelo': 'land_use_period', 'anio_inicial_suelo': 'land_use_initial_year', 'anio_final_suelo': 'land_use_final_year', 'clase': 'class', 'nombre_clase': 'class_name', 'area_total_zona_ha': 'total_zone_area_ha', 'area_inicial_clase_ha': 'initial_class_area_ha', 'area_final_clase_ha': 'final_class_area_ha', 'persistencia_ha': 'persistence_ha', 'perdida_ha': 'loss_ha', 'ganancia_ha': 'gain_ha', 'cambio_neto_ha': 'net_change_ha', 'cambio_total_ha': 'total_change_ha', 'ganancia_pp': 'gain_pp', 'perdida_pp': 'loss_pp', 'recambio_pp': 'turnover_pp', 'escenario': 'scenario', 'parcial_r_periodo': 'partial_r_period', 'parcial_p': 'partial_p', 'parcial_p_ajustado_BH': 'partial_p_adjusted_BH', 'significativa_BH_0_05': 'significant_BH_0_05', 'area_ha': 'area_ha', 'fuente': 'source', 'superficie_total_ha': 'total_area_ha', 'porcentaje': 'percentage', 'diferencia_ha': 'difference_ha', 'diferencia_absoluta_ha': 'absolute_difference_ha', 'escenario_nombre': 'scenario_name', 'clase_nombre': 'class_name', 'parametro_nombre': 'parameter_name', 'parcial_p_BH_nuevo': 'partial_p_BH'}

def for_export(table):
    return table.rename(columns=OUTPUT_COLUMNS_EN)

def exportar_datos(
    calidad_cruda,
    resumen
):

    ruta_excel = (
        CARPETA_SALIDA
        / "Water_quality_time_series_data.xlsx"
    )

    with pd.ExcelWriter(
        ruta_excel,
        engine="openpyxl"
    ) as writer:

        for_export(calidad_cruda).to_excel(
            writer,
            sheet_name="Original_estimates",
            index=False
        )

        for_export(resumen).to_excel(
            writer,
            sheet_name="Annual_summary",
            index=False
        )

        for parametro in [
            "COT",
            "SDT",
            "Cl-a"
        ]:

            tabla_parametro = resumen[
                resumen["parametro"] == parametro
            ].copy()

            for_export(tabla_parametro).to_excel(
                writer,
                sheet_name=f"Series_{parametro}",
                index=False
            )

    return ruta_excel


# ============================================================
# 15. EJECUCIÓN DEL ANÁLISIS
# ============================================================

verificar_archivos()

print(
    "\nLeyendo las estimaciones de calidad del agua..."
)

calidad_cruda = leer_archivos_calidad()

print(
    "Registros originales:",
    len(calidad_cruda)
)

print(
    "\nCalculando los resúmenes anuales..."
)

resumen_anual = calcular_resumen_anual(
    calidad_cruda
)

print(
    "Resúmenes anuales:",
    len(resumen_anual)
)


# ============================================================
# 16. COMPROBACIÓN DE LA ESTRUCTURA
# ============================================================

comprobacion = (
    calidad_cruda
    .groupby(
        [
            "parametro",
            "anio",
            "punto"
        ]
    )
    .size()
    .reset_index(
        name="numero_observaciones"
    )
)

print(
    "\nNúmero de observaciones por parámetro, año y punto:"
)

display(
    comprobacion
)

if not (
    comprobacion["numero_observaciones"] == 2
).all():

    print(
        "\nAdvertencia: no todas las combinaciones "
        "tienen exactamente dos estimaciones."
    )

else:

    print(
        "\nTodas las combinaciones contienen "
        "exactamente dos estimaciones."
    )


# ============================================================
# 17. PRESENTACIÓN DEL RESUMEN ANUAL
# ============================================================

print(
    "\nResumen anual utilizado en las gráficas:"
)

display(
    resumen_anual.round(4)
)


# ============================================================
# 18. CREACIÓN DE LA FIGURA COMBINADA
# ============================================================

print(
    "\nGenerating combined figure..."
)

rutas_combinadas = crear_figura_combinada(
    resumen_anual
)


# ============================================================
# 19. CREACIÓN DE FIGURAS INDIVIDUALES
# ============================================================

print(
    "\nGenerating individual figures..."
)

rutas_individuales = crear_figuras_individuales(
    resumen_anual
)


# ============================================================
# 20. EXPORTACIÓN A EXCEL
# ============================================================

ruta_excel = exportar_datos(
    calidad_cruda,
    resumen_anual
)


# ============================================================
# 21. MENSAJE FINAL
# ============================================================

print("\n" + "=" * 68)

print(
    "PROCESO TERMINADO CORRECTAMENTE"
)

print("=" * 68)

print(
    "\nCombined figure:"
)

for ruta in rutas_combinadas:
    print(ruta)

print(
    "\nArchivo Excel:"
)

print(ruta_excel)

print(
    "\nAll results were saved in:"
)

print(CARPETA_SALIDA)

In [ ]:
# ============================================================
# BARRAS APILADAS AL 100 %
# COMPOSICIÓN DE LAS COBERTURAS POR ZONA DE ESCURRIMIENTO
# AÑOS 2013, 2015 Y 2017
# ============================================================
#
# Clases:
# 1. Agua
# 2. Bosque
# 3. Agrícola
# 4. Matorral
# 5. Sin vegetación aparente
#
# Para 2015:
# - Se utiliza como valor principal el área final del archivo
#   correspondiente a 2013-2015.
# - Se compara contra el área inicial del archivo 2015-2017.
#
# Salidas:
# - Figura combinada de las seis zonas: PNG, PDF y SVG.
# - Una figura individual para cada zona.
# - Libro Excel con áreas y porcentajes.
# ============================================================


# ============================================================
# 1. INSTALACIÓN DE PAQUETES
# ============================================================



# ============================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.patches import Patch
from IPython.display import display


# ============================================================
# 3. CONEXIÓN CON GOOGLE DRIVE
# ============================================================



# ============================================================
# 4. CARPETAS
# ============================================================

CARPETA_DATOS = REPO_ROOT / "derived_data" / "lulc_change"

CARPETA_SALIDA = (
    REPO_ROOT / "results" / "correlation" / "Stacked_land_cover_figures"
)

CARPETA_SALIDA.mkdir(
    parents=True,
    exist_ok=True
)

print("Data folder:")
print(CARPETA_DATOS)

print("\nOutput folder:")
print(CARPETA_SALIDA)


# ============================================================
# 5. NOMBRES DE LAS CLASES
# ============================================================

NOMBRES_CLASES = {
    1: "Water",
    2: "Forest",
    3: "Agricultural land",
    4: "Tropical forests",
    5: "Bare land",
}


# ============================================================
# 6. COLORES DE LAS CLASES
# ============================================================
#
# Se utilizan colores semejantes a los de la reclasificación:
# Agua: azul
# Bosque: verde oscuro
# Agrícola: naranja
# Matorral: verde claro
# Sin vegetación aparente: gris
# ============================================================

COLORES_CLASES = {
    1: "#0000FF",
    2: "#087F23",
    3: "#FFA000",
    4: "#90EE90",
    5: "#8C8C8C",
}


# Color del texto dentro de cada cobertura.
COLORES_TEXTO = {
    1: "white",
    2: "white",
    3: "black",
    4: "black",
    5: "white",
}


# ============================================================
# 7. ARCHIVOS DE CAMBIO DE USO DEL SUELO
# ============================================================

ARCHIVOS_SUELO = {
    (2013, 2015, 1):
        "land_use_change_analysis_2013_2015_zone_1.xlsx",

    (2013, 2015, 2):
        "land_use_change_analysis_2013_2015_zone_2.xlsx",

    (2013, 2015, 3):
        "land_use_change_analysis_2013_2015_zone_3.xlsx",

    (2013, 2015, 4):
        "land_use_change_analysis_2013_2015_zone_4.xlsx",

    (2013, 2015, 5):
        "land_use_change_analysis_2013_2015_zone_5.xlsx",

    (2013, 2015, 6):
        "land_use_change_analysis_2013_2015_zone_6.xlsx",

    (2015, 2017, 1):
        "land_use_change_analysis_2015_2017_zone_1.xlsx",

    (2015, 2017, 2):
        "land_use_change_analysis_2015_2017_zone_2.xlsx",

    (2015, 2017, 3):
        "land_use_change_analysis_2015_2017_zone_3.xlsx",

    (2015, 2017, 4):
        "land_use_change_analysis_2015_2017_zone_4.xlsx",

    (2015, 2017, 5):
        "land_use_change_analysis_2015_2017_zone_5.xlsx",

    (2015, 2017, 6):
        "land_use_change_analysis_2015_2017_zone_6.xlsx",
}


# ============================================================
# 8. CONFIGURACIÓN GENERAL DE LAS GRÁFICAS
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "axes.linewidth": 0.8,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 120,
    "savefig.dpi": 600,
})


# ============================================================
# 9. VERIFICACIÓN DE ARCHIVOS
# ============================================================

def verificar_archivos():

    faltantes = []

    for nombre in ARCHIVOS_SUELO.values():

        ruta = CARPETA_DATOS / nombre

        if not ruta.exists():
            faltantes.append(nombre)

    print(
        "\nExpected files:",
        len(ARCHIVOS_SUELO)
    )

    print(
        "Files found:",
        len(ARCHIVOS_SUELO) - len(faltantes)
    )

    if faltantes:

        print(
            "\nThe following files were not found:"
        )

        for nombre in faltantes:
            print(" -", nombre)

        raise FileNotFoundError(
            "Faltan archivos de uso del suelo."
        )

    print(
        "\nTodos los archivos de uso del suelo "
        "fueron encontrados correctamente."
    )


# ============================================================
# 10. FUNCIÓN PARA LEER UN RESUMEN DE CAMBIO
# ============================================================

def leer_resumen(nombre_archivo):

    ruta = CARPETA_DATOS / nombre_archivo

    datos = pd.read_excel(
        ruta,
        sheet_name="Change_summary"
    )

    columnas_necesarias = [
        "Class",
        "Initial Area (ha)",
        "Final Area (ha)"
    ]

    faltantes = [
        columna
        for columna in columnas_necesarias
        if columna not in datos.columns
    ]

    if faltantes:

        raise ValueError(
            f"Faltan columnas en {nombre_archivo}: "
            f"{faltantes}"
        )

    datos = datos[
        columnas_necesarias
    ].copy()

    datos["Class"] = (
        datos["Class"].astype(int)
    )

    return datos


# ============================================================
# 11. CONSTRUCCIÓN DE LAS COBERTURAS 2013, 2015 Y 2017
# ============================================================

def construir_tabla_coberturas():

    registros = []
    comprobaciones_2015 = []

    for zona in range(1, 7):

        archivo_2013_2015 = (
            ARCHIVOS_SUELO[
                (2013, 2015, zona)
            ]
        )

        archivo_2015_2017 = (
            ARCHIVOS_SUELO[
                (2015, 2017, zona)
            ]
        )

        datos_2013_2015 = leer_resumen(
            archivo_2013_2015
        )

        datos_2015_2017 = leer_resumen(
            archivo_2015_2017
        )

        datos_2013_2015 = (
            datos_2013_2015
            .set_index("Class")
        )

        datos_2015_2017 = (
            datos_2015_2017
            .set_index("Class")
        )

        for clase in range(1, 6):

            # 2013: área inicial del periodo 2013-2015.
            area_2013 = datos_2013_2015.loc[
                clase,
                "Initial Area (ha)"
            ]

            # 2015 principal: área final del periodo 2013-2015.
            area_2015 = datos_2013_2015.loc[
                clase,
                "Final Area (ha)"
            ]

            # 2015 de comprobación:
            # área inicial del periodo 2015-2017.
            area_2015_comprobacion = (
                datos_2015_2017.loc[
                    clase,
                    "Initial Area (ha)"
                ]
            )

            # 2017: área final del periodo 2015-2017.
            area_2017 = datos_2015_2017.loc[
                clase,
                "Final Area (ha)"
            ]

            diferencia_2015 = (
                area_2015_comprobacion -
                area_2015
            )

            comprobaciones_2015.append({
                "zona": zona,
                "clase": clase,
                "nombre_clase":
                    NOMBRES_CLASES[clase],
                "area_2015_final_2013_2015_ha":
                    area_2015,
                "area_2015_inicial_2015_2017_ha":
                    area_2015_comprobacion,
                "diferencia_ha":
                    diferencia_2015,
                "diferencia_absoluta_ha":
                    abs(diferencia_2015)
            })

            registros.extend([
                {
                    "zona": zona,
                    "anio": 2013,
                    "clase": clase,
                    "nombre_clase":
                        NOMBRES_CLASES[clase],
                    "area_ha": area_2013,
                    "fuente":
                        "Área inicial 2013-2015"
                },
                {
                    "zona": zona,
                    "anio": 2015,
                    "clase": clase,
                    "nombre_clase":
                        NOMBRES_CLASES[clase],
                    "area_ha": area_2015,
                    "fuente":
                        "Área final 2013-2015"
                },
                {
                    "zona": zona,
                    "anio": 2017,
                    "clase": clase,
                    "nombre_clase":
                        NOMBRES_CLASES[clase],
                    "area_ha": area_2017,
                    "fuente":
                        "Área final 2015-2017"
                }
            ])

    coberturas = pd.DataFrame(
        registros
    )

    comprobacion_2015 = pd.DataFrame(
        comprobaciones_2015
    )

    # Superficie total de cada zona y año.
    totales = (
        coberturas
        .groupby(
            ["zona", "anio"],
            as_index=False
        )
        .agg(
            superficie_total_ha=(
                "area_ha",
                "sum"
            )
        )
    )

    coberturas = coberturas.merge(
        totales,
        on=["zona", "anio"],
        how="left",
        validate="many_to_one"
    )

    # Porcentaje de cada cobertura dentro de la zona.
    coberturas["porcentaje"] = (
        100
        * coberturas["area_ha"]
        / coberturas["superficie_total_ha"]
    )

    # Comprobación de que cada barra suma 100 %.
    sumas_porcentaje = (
        coberturas
        .groupby(
            ["zona", "anio"],
            as_index=False
        )
        .agg(
            suma_porcentaje=(
                "porcentaje",
                "sum"
            )
        )
    )

    return (
        coberturas,
        totales,
        comprobacion_2015,
        sumas_porcentaje
    )


# ============================================================
# 12. FUNCIÓN PARA AGREGAR ETIQUETAS A LAS BARRAS
# ============================================================

def agregar_etiqueta_segmento(
    eje,
    posicion_x,
    base,
    altura,
    clase,
    limite_etiqueta=4.0
):

    # Solo se muestran etiquetas cuando la cobertura
    # representa al menos 4 % de la zona.
    if altura >= limite_etiqueta:

        eje.text(
            posicion_x,
            base + altura / 2,
            f"{altura:.1f}",
            ha="center",
            va="center",
            fontsize=10,
            color=COLORES_TEXTO[clase],
            fontweight="normal",
            clip_on=True
        )


# ============================================================
# 13. FIGURA COMBINADA DE LAS SEIS ZONAS
# ============================================================

def crear_figura_combinada(coberturas):

    figura, ejes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(13.5, 8.5),
        sharey=True
    )

    ejes = ejes.flatten()

    anios = [2013, 2015, 2017]
    posiciones = np.arange(len(anios))

    for zona, eje in zip(
        range(1, 7),
        ejes
    ):

        datos_zona = coberturas[
            coberturas["zona"] == zona
        ].copy()

        base = np.zeros(
            len(anios)
        )

        for clase in range(1, 6):

            porcentajes = []

            for anio in anios:

                seleccion = datos_zona[
                    (
                        datos_zona["anio"] == anio
                    )
                    &
                    (
                        datos_zona["clase"] == clase
                    )
                ]

                porcentaje = (
                    seleccion["porcentaje"].iloc[0]
                )

                porcentajes.append(
                    porcentaje
                )

            porcentajes = np.array(
                porcentajes
            )

            eje.bar(
                posiciones,
                porcentajes,
                bottom=base,
                width=0.66,
                color=COLORES_CLASES[clase],
                edgecolor="white",
                linewidth=0.7,
                label=NOMBRES_CLASES[clase]
            )

            for indice, porcentaje in enumerate(
                porcentajes
            ):

                agregar_etiqueta_segmento(
                    eje=eje,
                    posicion_x=posiciones[indice],
                    base=base[indice],
                    altura=porcentaje,
                    clase=clase,
                    limite_etiqueta=4.0
                )

            base = base + porcentajes

        eje.set_title(
            f"Runoff zone {zona}",
            fontweight="normal",
            pad=8
        )

        eje.set_xticks(
            posiciones
        )

        eje.set_xticklabels(
            anios
        )

        eje.set_ylim(
            0,
            100
        )

        eje.set_yticks(
            np.arange(0, 101, 20)
        )

        eje.grid(
            axis="y",
            linestyle="--",
            linewidth=0.5,
            alpha=0.35
        )

        eje.set_axisbelow(True)

        eje.spines["top"].set_visible(False)
        eje.spines["right"].set_visible(False)

        eje.tick_params(
            axis="both",
            direction="out"
        )

    # Etiquetas del eje vertical en la primera columna.
    ejes[0].set_ylabel(
        "Zone area (%)",
        fontweight="normal"
    )

    ejes[3].set_ylabel(
        "Zone area (%)",
        fontweight="normal"
    )

    # Etiquetas del eje horizontal en la fila inferior.
    for eje in ejes[3:]:
        eje.set_xlabel(
            "Year",
            fontweight="normal"
        )

    # Leyenda compartida.
    elementos_leyenda = [
        Patch(
            facecolor=COLORES_CLASES[clase],
            edgecolor="none",
            label=NOMBRES_CLASES[clase]
        )
        for clase in range(1, 6)
    ]

    figura.legend(
        handles=elementos_leyenda,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=5,
        frameon=False,
        columnspacing=1.6,
        handlelength=1.5,
        handletextpad=0.5
    )

    figura.suptitle(
        (
            "Land-cover composition "
            "by runoff zone"
        ),
        fontsize=16,
        fontweight="normal",
        y=0.99
    )

    figura.tight_layout(
        rect=[0, 0.07, 1, 0.96],
        h_pad=2.0,
        w_pad=1.4
    )

    ruta_png = (
        CARPETA_SALIDA
        / "Stacked_land_cover_100_percent_by_zone.png"
    )

    ruta_pdf = (
        CARPETA_SALIDA
        / "Stacked_land_cover_100_percent_by_zone.pdf"
    )

    ruta_svg = (
        CARPETA_SALIDA
        / "Stacked_land_cover_100_percent_by_zone.svg"
    )

    figura.savefig(
        ruta_png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_svg,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    return (
        ruta_png,
        ruta_pdf,
        ruta_svg
    )


# ============================================================
# 14. FIGURAS INDIVIDUALES POR ZONA
# ============================================================

def crear_figuras_individuales(coberturas):

    rutas = []

    anios = [2013, 2015, 2017]
    posiciones = np.arange(len(anios))

    for zona in range(1, 7):

        datos_zona = coberturas[
            coberturas["zona"] == zona
        ].copy()

        figura, eje = plt.subplots(
            figsize=(7.5, 6)
        )

        base = np.zeros(
            len(anios)
        )

        for clase in range(1, 6):

            porcentajes = []

            for anio in anios:

                seleccion = datos_zona[
                    (
                        datos_zona["anio"] == anio
                    )
                    &
                    (
                        datos_zona["clase"] == clase
                    )
                ]

                porcentajes.append(
                    seleccion["porcentaje"].iloc[0]
                )

            porcentajes = np.array(
                porcentajes
            )

            eje.bar(
                posiciones,
                porcentajes,
                bottom=base,
                width=0.60,
                color=COLORES_CLASES[clase],
                edgecolor="white",
                linewidth=0.8,
                label=NOMBRES_CLASES[clase]
            )

            for indice, porcentaje in enumerate(
                porcentajes
            ):

                agregar_etiqueta_segmento(
                    eje=eje,
                    posicion_x=posiciones[indice],
                    base=base[indice],
                    altura=porcentaje,
                    clase=clase,
                    limite_etiqueta=3.0
                )

            base = base + porcentajes

        eje.set_title(
            (
                "Land-cover composition "
                f"in runoff zone {zona}"
            ),
            fontweight="normal",
            pad=10
        )

        eje.set_xlabel(
            "Year",
            fontweight="normal"
        )

        eje.set_ylabel(
            "Zone area (%)",
            fontweight="normal"
        )

        eje.set_xticks(
            posiciones
        )

        eje.set_xticklabels(
            anios
        )

        eje.set_ylim(
            0,
            100
        )

        eje.set_yticks(
            np.arange(0, 101, 10)
        )

        eje.grid(
            axis="y",
            linestyle="--",
            linewidth=0.5,
            alpha=0.35
        )

        eje.set_axisbelow(True)

        eje.spines["top"].set_visible(False)
        eje.spines["right"].set_visible(False)

        eje.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.12),
            ncol=3,
            frameon=False,
            columnspacing=1.4,
            handletextpad=0.5
        )

        figura.tight_layout()

        ruta_png = (
            CARPETA_SALIDA
            / f"Stacked_land_cover_zone_{zona}.png"
        )

        ruta_pdf = (
            CARPETA_SALIDA
            / f"Stacked_land_cover_zone_{zona}.pdf"
        )

        figura.savefig(
            ruta_png,
            dpi=600,
            bbox_inches="tight",
            facecolor="white"
        )

        figura.savefig(
            ruta_pdf,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()

        rutas.extend([
            ruta_png,
            ruta_pdf
        ])

    return rutas


# ============================================================
# 15. TABLA EN FORMATO ANCHO
# ============================================================

def crear_tabla_ancha(coberturas):

    tabla_ancha = (
        coberturas
        .pivot_table(
            index=[
                "zona",
                "anio"
            ],
            columns="nombre_clase",
            values="porcentaje"
        )
        .reset_index()
    )

    tabla_ancha.columns.name = None

    orden_columnas = [
        "zona",
        "anio",
        "Water",
        "Forest",
        "Agricultural land",
        "Tropical forests",
        "Bare land"
    ]

    tabla_ancha = tabla_ancha[
        orden_columnas
    ]

    tabla_ancha["Total"] = (
        tabla_ancha[
            [
                "Water",
                "Forest",
                "Agricultural land",
                "Tropical forests",
                "Bare land"
            ]
        ].sum(axis=1)
    )

    return tabla_ancha


# ============================================================
# 16. EXPORTACIÓN A EXCEL
# ============================================================


# English headers used in every exported table.
OUTPUT_COLUMNS_EN = {'anio': 'year', 'punto': 'point', 'zona': 'zone', 'parametro': 'parameter', 'estimacion': 'estimate', 'archivo_origen': 'source_file', 'observacion': 'observation', 'numero_estimaciones': 'number_of_estimates', 'media_geometrica': 'geometric_mean', 'media_aritmetica': 'arithmetic_mean', 'mediana': 'median', 'minimo': 'minimum', 'maximo': 'maximum', 'desviacion_estandar': 'standard_deviation', 'error_inferior': 'lower_error', 'error_superior': 'upper_error', 'periodo_calidad': 'water_quality_period', 'anio_inicial_calidad': 'water_quality_initial_year', 'anio_final_calidad': 'water_quality_final_year', 'valor_inicial': 'initial_value', 'valor_final': 'final_value', 'cambio_log': 'log_change', 'cambio_porcentual': 'percent_change', 'periodo_suelo': 'land_use_period', 'anio_inicial_suelo': 'land_use_initial_year', 'anio_final_suelo': 'land_use_final_year', 'clase': 'class', 'nombre_clase': 'class_name', 'area_total_zona_ha': 'total_zone_area_ha', 'area_inicial_clase_ha': 'initial_class_area_ha', 'area_final_clase_ha': 'final_class_area_ha', 'persistencia_ha': 'persistence_ha', 'perdida_ha': 'loss_ha', 'ganancia_ha': 'gain_ha', 'cambio_neto_ha': 'net_change_ha', 'cambio_total_ha': 'total_change_ha', 'ganancia_pp': 'gain_pp', 'perdida_pp': 'loss_pp', 'recambio_pp': 'turnover_pp', 'escenario': 'scenario', 'parcial_r_periodo': 'partial_r_period', 'parcial_p': 'partial_p', 'parcial_p_ajustado_BH': 'partial_p_adjusted_BH', 'significativa_BH_0_05': 'significant_BH_0_05', 'area_ha': 'area_ha', 'fuente': 'source', 'superficie_total_ha': 'total_area_ha', 'porcentaje': 'percentage', 'diferencia_ha': 'difference_ha', 'diferencia_absoluta_ha': 'absolute_difference_ha', 'escenario_nombre': 'scenario_name', 'clase_nombre': 'class_name', 'parametro_nombre': 'parameter_name', 'parcial_p_BH_nuevo': 'partial_p_BH'}

def for_export(table):
    return table.rename(columns=OUTPUT_COLUMNS_EN)

def exportar_resultados(
    coberturas,
    totales,
    comprobacion_2015,
    sumas_porcentaje,
    tabla_ancha
):

    ruta_excel = (
        CARPETA_SALIDA
        / "Stacked_land_cover_data_by_zone.xlsx"
    )

    with pd.ExcelWriter(
        ruta_excel,
        engine="openpyxl"
    ) as writer:

        for_export(coberturas).to_excel(
            writer,
            sheet_name="Land_cover_long_format",
            index=False
        )

        for_export(tabla_ancha).to_excel(
            writer,
            sheet_name="Percentages_by_zone",
            index=False
        )

        for_export(totales).to_excel(
            writer,
            sheet_name="Total_areas",
            index=False
        )

        for_export(comprobacion_2015).to_excel(
            writer,
            sheet_name="Check_2015",
            index=False
        )

        for_export(sumas_porcentaje).to_excel(
            writer,
            sheet_name="Check_100_percent",
            index=False
        )

    return ruta_excel


# ============================================================
# 17. EJECUCIÓN
# ============================================================

verificar_archivos()

print(
    "\nConstruyendo las coberturas de "
    "2013, 2015 y 2017..."
)

(
    coberturas,
    superficies_totales,
    comprobacion_2015,
    sumas_porcentaje
) = construir_tabla_coberturas()


# ============================================================
# 18. COMPROBACIÓN DEL AÑO 2015
# ============================================================

maxima_diferencia_2015 = (
    comprobacion_2015[
        "diferencia_absoluta_ha"
    ].max()
)

print(
    "\nMáxima diferencia entre las dos "
    "fuentes disponibles para 2015:"
)

print(
    f"{maxima_diferencia_2015:.4f} ha"
)

if maxima_diferencia_2015 <= 1:

    print(
        "Las dos representaciones de 2015 "
        "son consistentes."
    )

else:

    print(
        "Advertencia: existe una diferencia superior "
        "a 1 ha en alguna clase de 2015."
    )

display(
    comprobacion_2015.round(4)
)


# ============================================================
# 19. COMPROBACIÓN DEL 100 %
# ============================================================

print(
    "\nComprobación de que cada zona y año "
    "suma 100 %:"
)

display(
    sumas_porcentaje.round(8)
)

if np.allclose(
    sumas_porcentaje["suma_porcentaje"],
    100,
    atol=0.0001
):

    print(
        "\nTodas las composiciones suman 100 %."
    )

else:

    print(
        "\nAdvertencia: alguna composición "
        "no suma exactamente 100 %."
    )


# ============================================================
# 20. TABLA FINAL DE PORCENTAJES
# ============================================================

tabla_porcentajes = crear_tabla_ancha(
    coberturas
)

print(
    "\nPorcentaje de las coberturas "
    "por zona y año:"
)

display(
    tabla_porcentajes.round(2)
)


# ============================================================
# 21. CREACIÓN DE LA FIGURA COMBINADA
# ============================================================

print(
    "\nGenerating combined figure..."
)

rutas_combinadas = crear_figura_combinada(
    coberturas
)


# ============================================================
# 22. CREACIÓN DE FIGURAS INDIVIDUALES
# ============================================================

print(
    "\nGenerating individual figures..."
)

rutas_individuales = (
    crear_figuras_individuales(
        coberturas
    )
)


# ============================================================
# 23. EXPORTACIÓN DE RESULTADOS
# ============================================================

ruta_excel = exportar_resultados(
    coberturas,
    superficies_totales,
    comprobacion_2015,
    sumas_porcentaje,
    tabla_porcentajes
)


# ============================================================
# 24. MENSAJE FINAL
# ============================================================

print("\n" + "=" * 68)

print(
    "PROCESO TERMINADO CORRECTAMENTE"
)

print("=" * 68)

print(
    "\nCombined figure:"
)

for ruta in rutas_combinadas:
    print(ruta)

print(
    "\nLibro Excel:"
)

print(ruta_excel)

print(
    "\nAll results were saved in:"
)

print(CARPETA_SALIDA)

In [ ]:
# ============================================================
# GRÁFICAS DE DISPERSIÓN
# RELACIONES ENTRE CAMBIO DE COBERTURA Y CALIDAD DEL AGUA
# ============================================================
#
# Exact temporal scenario:
# 2013-2015 y 2015-2017
#
# Cada punto representa:
# zona de escurrimiento × periodo
#
# Eje horizontal:
# Cambio neto de la cobertura en puntos porcentuales.
#
# Eje vertical:
# Cambio logarítmico del parámetro:
# 100 × ln(valor final / valor inicial)
#
# Las líneas representan regresiones ajustadas por periodo,
# con una pendiente común y diferente intercepto temporal.
# ============================================================


# ============================================================
# 1. INSTALACIÓN DE PAQUETES
# ============================================================



# ============================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

from pathlib import Path

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import stats
from matplotlib.lines import Line2D
from IPython.display import display


# ============================================================
# 3. CONEXIÓN CON GOOGLE DRIVE
# ============================================================



# ============================================================
# 4. CARPETAS
# ============================================================

CARPETA_PRINCIPAL = REPO_ROOT / "results" / "correlation"

ARCHIVO_RESULTADOS = (
    CARPETA_PRINCIPAL / "Unified_land_use_water_quality_results.xlsx"
)

CARPETA_SALIDA = (
    CARPETA_PRINCIPAL / "Scatterplot_figures"
)

CARPETA_SALIDA.mkdir(
    parents=True,
    exist_ok=True
)

print("Input file:")
print(ARCHIVO_RESULTADOS)

print("\nOutput folder:")
print(CARPETA_SALIDA)


# ============================================================
# 5. NOMBRES DE LAS CLASES
# ============================================================

NOMBRES_CLASES = {
    1: "Water",
    2: "Forest",
    3: "Agricultural land",
    4: "Tropical forests",
    5: "Bare land",
}


# ============================================================
# 6. RELACIONES QUE SE GRAFICARÁN
# ============================================================

RELACIONES_RELEVANTES = [
    {
        "clase": 3,
        "parametro": "COT",
        "titulo": "Agricultural land and TOC"
    },
    {
        "clase": 1,
        "parametro": "COT",
        "titulo": "Water and TOC"
    },
    {
        "clase": 5,
        "parametro": "COT",
        "titulo": "Bare land and TOC"
    },
    {
        "clase": 5,
        "parametro": "SDT",
        "titulo": "Bare land and TDS"
    },
    {
        "clase": 3,
        "parametro": "Cl_a",
        "titulo": "Agricultural land and Chl-a"
    },
]


# ============================================================
# 7. NOMBRES Y UNIDADES DE LOS PARÁMETROS
# ============================================================

NOMBRES_PARAMETROS = {
    "COT": "TOC",
    "SDT": "TDS",
    "Cl_a": "Chl-a",
    "Cl-a": "Chl-a",
}

UNIDADES_PARAMETROS = {
    "COT": "Logarithmic change in TOC (%)",
    "SDT": "Logarithmic change in TDS (%)",
    "Cl_a": "Logarithmic change in Chl-a (%)",
    "Cl-a": "Logarithmic change in Chl-a (%)",
}


# ============================================================
# 8. COLORES Y SÍMBOLOS DE LOS PERIODOS
# ============================================================

COLORES_PERIODOS = {
    "2013-2015": "#1F77B4",
    "2015-2017": "#D62728",
}

MARCADORES_PERIODOS = {
    "2013-2015": "o",
    "2015-2017": "s",
}


# ============================================================
# 9. CONFIGURACIÓN GENERAL DE LAS GRÁFICAS
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "axes.linewidth": 0.8,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 120,
    "savefig.dpi": 600,
})


# ============================================================
# 10. VERIFICACIÓN Y LECTURA DEL ARCHIVO
# ============================================================

if not ARCHIVO_RESULTADOS.exists():

    raise FileNotFoundError(
        "No se encontró el archivo de resultados:\n"
        f"{ARCHIVO_RESULTADOS}\n\n"
        "Primero debes ejecutar el código de correlación."
    )

tabla_exacta = pd.read_excel(
    ARCHIVO_RESULTADOS,
    sheet_name="Exact_analysis_table"
)

INPUT_COLUMNS_INTERNAL = {'year': 'anio', 'point': 'punto', 'zone': 'zona', 'parameter': 'parametro', 'estimate': 'estimacion', 'source_file': 'archivo_origen', 'observation': 'observacion', 'number_of_estimates': 'numero_estimaciones', 'geometric_mean': 'media_geometrica', 'arithmetic_mean': 'media_aritmetica', 'median': 'mediana', 'minimum': 'minimo', 'maximum': 'maximo', 'standard_deviation': 'desviacion_estandar', 'lower_error': 'error_inferior', 'upper_error': 'error_superior', 'water_quality_period': 'periodo_calidad', 'water_quality_initial_year': 'anio_inicial_calidad', 'water_quality_final_year': 'anio_final_calidad', 'initial_value': 'valor_inicial', 'final_value': 'valor_final', 'log_change': 'cambio_log', 'percent_change': 'cambio_porcentual', 'land_use_period': 'periodo_suelo', 'land_use_initial_year': 'anio_inicial_suelo', 'land_use_final_year': 'anio_final_suelo', 'class': 'clase', 'class_name': 'clase_nombre', 'total_zone_area_ha': 'area_total_zona_ha', 'initial_class_area_ha': 'area_inicial_clase_ha', 'final_class_area_ha': 'area_final_clase_ha', 'persistence_ha': 'persistencia_ha', 'loss_ha': 'perdida_ha', 'gain_ha': 'ganancia_ha', 'net_change_ha': 'cambio_neto_ha', 'total_change_ha': 'cambio_total_ha', 'gain_pp': 'ganancia_pp', 'loss_pp': 'perdida_pp', 'turnover_pp': 'recambio_pp', 'scenario': 'escenario', 'partial_r_period': 'parcial_r_periodo', 'partial_p': 'parcial_p', 'partial_p_adjusted_BH': 'parcial_p_ajustado_BH', 'significant_BH_0_05': 'significativa_BH_0_05', 'area_ha': 'area_ha', 'source': 'fuente', 'total_area_ha': 'superficie_total_ha', 'percentage': 'porcentaje', 'difference_ha': 'diferencia_ha', 'absolute_difference_ha': 'diferencia_absoluta_ha', 'scenario_name': 'escenario_nombre', 'parameter_name': 'parametro_nombre', 'partial_p_BH': 'parcial_p_BH_nuevo'}
tabla_exacta = tabla_exacta.rename(columns=INPUT_COLUMNS_INTERNAL)


print(
    "\nRegistros encontrados en la tabla analítica:",
    len(tabla_exacta)
)

print("\nAvailable columns:")

for columna in tabla_exacta.columns:
    print(" -", columna)


# ============================================================
# 11. NORMALIZACIÓN DE COLUMNAS
# ============================================================

columnas_necesarias = [
    "zona",
    "periodo_suelo",
    "clase",
    "parametro",
    "delta_pp",
    "cambio_log"
]

columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in tabla_exacta.columns
]

if columnas_faltantes:

    raise ValueError(
        "Faltan las siguientes columnas en "
        f"Tabla_analitica_exacta: {columnas_faltantes}"
    )

tabla_exacta["zona"] = (
    tabla_exacta["zona"].astype(int)
)

tabla_exacta["clase"] = (
    tabla_exacta["clase"].astype(int)
)

tabla_exacta["periodo_suelo"] = (
    tabla_exacta["periodo_suelo"].astype(str)
)

tabla_exacta["parametro"] = (
    tabla_exacta["parametro"]
    .replace({
        "Cl-a": "Cl_a",
        "Cha-a": "Cl_a"
    })
)

tabla_exacta["nombre_clase"] = (
    tabla_exacta["clase"]
    .map(NOMBRES_CLASES)
)

tabla_exacta["etiqueta_zona"] = (
    "Z" + tabla_exacta["zona"].astype(str)
)


# ============================================================
# 12. CORRELACIÓN PARCIAL CONTROLANDO EL PERIODO
# ============================================================

def correlacion_parcial_periodo(datos):

    residuo_x = (
        datos["delta_pp"]
        -
        datos.groupby("periodo_suelo")[
            "delta_pp"
        ].transform("mean")
    )

    residuo_y = (
        datos["cambio_log"]
        -
        datos.groupby("periodo_suelo")[
            "cambio_log"
        ].transform("mean")
    )

    resultado = stats.pearsonr(
        residuo_x,
        residuo_y
    )

    return (
        resultado.statistic,
        resultado.pvalue
    )


# ============================================================
# 13. REGRESIÓN AJUSTADA POR PERIODO
# ============================================================

def regresion_ajustada_periodo(datos):

    periodos = sorted(
        datos["periodo_suelo"].unique()
    )

    if len(periodos) != 2:

        raise ValueError(
            "Se esperaban exactamente dos periodos."
        )

    indicador_periodo = (
        datos["periodo_suelo"] == periodos[1]
    ).astype(float).to_numpy()

    x = datos["delta_pp"].to_numpy()
    y = datos["cambio_log"].to_numpy()

    matriz_x = np.column_stack([
        np.ones(len(datos)),
        x,
        indicador_periodo
    ])

    coeficientes = np.linalg.lstsq(
        matriz_x,
        y,
        rcond=None
    )[0]

    intercepto = coeficientes[0]
    pendiente = coeficientes[1]
    efecto_periodo = coeficientes[2]

    predicciones = matriz_x @ coeficientes
    residuos = y - predicciones

    numero_parametros = matriz_x.shape[1]
    grados_libertad = len(y) - numero_parametros

    suma_cuadrados_residuos = (
        residuos @ residuos
    )

    varianza_residual = (
        suma_cuadrados_residuos
        / grados_libertad
    )

    covarianza = (
        varianza_residual
        *
        np.linalg.inv(
            matriz_x.T @ matriz_x
        )
    )

    error_estandar_pendiente = (
        np.sqrt(
            np.diag(covarianza)
        )[1]
    )

    estadistico_t = (
        pendiente
        / error_estandar_pendiente
    )

    valor_p = (
        2
        * stats.t.sf(
            abs(estadistico_t),
            grados_libertad
        )
    )

    valor_critico = stats.t.ppf(
        0.975,
        grados_libertad
    )

    intervalo_inferior = (
        pendiente
        -
        valor_critico
        * error_estandar_pendiente
    )

    intervalo_superior = (
        pendiente
        +
        valor_critico
        * error_estandar_pendiente
    )

    return {
        "periodos": periodos,
        "intercepto": intercepto,
        "pendiente": pendiente,
        "efecto_periodo": efecto_periodo,
        "error_estandar": error_estandar_pendiente,
        "estadistico_t": estadistico_t,
        "valor_p": valor_p,
        "ic95_inferior": intervalo_inferior,
        "ic95_superior": intervalo_superior
    }


# ============================================================
# 14. CÁLCULO DE ESTADÍSTICOS
# ============================================================

def calcular_estadisticos(datos):

    datos = datos.dropna(
        subset=[
            "delta_pp",
            "cambio_log"
        ]
    ).copy()

    resultado_pearson = stats.pearsonr(
        datos["delta_pp"],
        datos["cambio_log"]
    )

    resultado_spearman = stats.spearmanr(
        datos["delta_pp"],
        datos["cambio_log"]
    )

    parcial_r, parcial_p = (
        correlacion_parcial_periodo(
            datos
        )
    )

    regresion = regresion_ajustada_periodo(
        datos
    )

    return {
        "n": len(datos),
        "pearson_r":
            resultado_pearson.statistic,
        "pearson_p":
            resultado_pearson.pvalue,
        "spearman_rho":
            resultado_spearman.statistic,
        "spearman_p":
            resultado_spearman.pvalue,
        "parcial_r":
            parcial_r,
        "parcial_p":
            parcial_p,
        "pendiente_ajustada":
            regresion["pendiente"],
        "pendiente_p":
            regresion["valor_p"],
        "pendiente_ic95_inferior":
            regresion["ic95_inferior"],
        "pendiente_ic95_superior":
            regresion["ic95_superior"],
        "regresion":
            regresion
    }


# ============================================================
# 15. TEXTO ESTADÍSTICO PARA CADA PANEL
# ============================================================

def crear_texto_estadistico(estadisticos):

    texto = (
        f"Pearson: r = "
        f"{estadisticos['pearson_r']:.2f}, "
        f"p = {estadisticos['pearson_p']:.3f}\n"
        f"Spearman: ρ = "
        f"{estadisticos['spearman_rho']:.2f}, "
        f"p = {estadisticos['spearman_p']:.3f}\n"
        f"Parcial: r = "
        f"{estadisticos['parcial_r']:.2f}, "
        f"p = {estadisticos['parcial_p']:.3f}"
    )

    return texto


# ============================================================
# 16. FUNCIÓN PARA GRAFICAR UNA RELACIÓN
# ============================================================

def graficar_relacion(
    eje,
    datos,
    clase,
    parametro,
    titulo,
    mostrar_leyenda=False
):

    datos = datos[
        (
            datos["clase"] == clase
        )
        &
        (
            datos["parametro"] == parametro
        )
    ].copy()

    datos = datos.dropna(
        subset=[
            "delta_pp",
            "cambio_log"
        ]
    )

    if len(datos) != 12:

        print(
            f"Advertencia: {titulo} contiene "
            f"{len(datos)} observaciones, no 12."
        )

    estadisticos = calcular_estadisticos(
        datos
    )

    regresion = estadisticos[
        "regresion"
    ]

    periodos = regresion["periodos"]

    # Rango horizontal para las líneas.
    margen_x = (
        datos["delta_pp"].max()
        -
        datos["delta_pp"].min()
    ) * 0.08

    if margen_x == 0:
        margen_x = 0.5

    x_linea = np.linspace(
        datos["delta_pp"].min() - margen_x,
        datos["delta_pp"].max() + margen_x,
        100
    )

    for indice_periodo, periodo in enumerate(
        periodos
    ):

        datos_periodo = datos[
            datos["periodo_suelo"] == periodo
        ].copy()

        color = COLORES_PERIODOS[periodo]
        marcador = MARCADORES_PERIODOS[periodo]

        # Puntos.
        eje.scatter(
            datos_periodo["delta_pp"],
            datos_periodo["cambio_log"],
            s=65,
            marker=marcador,
            facecolor=color,
            edgecolor="white",
            linewidth=0.8,
            alpha=0.95,
            label=periodo,
            zorder=4
        )

        # Identificación de las zonas.
        for _, fila in datos_periodo.iterrows():

            eje.annotate(
                fila["etiqueta_zona"],
                xy=(
                    fila["delta_pp"],
                    fila["cambio_log"]
                ),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=10,
                color="#333333",
                fontweight="normal",
                zorder=5
            )

        # Línea ajustada con pendiente común.
        intercepto_periodo = (
            regresion["intercepto"]
            +
            indice_periodo
            * regresion["efecto_periodo"]
        )

        y_linea = (
            intercepto_periodo
            +
            regresion["pendiente"]
            * x_linea
        )

        eje.plot(
            x_linea,
            y_linea,
            color=color,
            linewidth=1.5,
            linestyle="-",
            alpha=0.85,
            zorder=2
        )

    # Líneas de referencia en cero.
    eje.axhline(
        0,
        color="#666666",
        linewidth=0.7,
        linestyle="--",
        alpha=0.55,
        zorder=1
    )

    eje.axvline(
        0,
        color="#666666",
        linewidth=0.7,
        linestyle="--",
        alpha=0.55,
        zorder=1
    )

    nombre_clase = NOMBRES_CLASES[
        clase
    ]

    nombre_parametro = NOMBRES_PARAMETROS[
        parametro
    ]

    eje.set_title(
        titulo,
        fontweight="normal",
        pad=8
    )

    eje.set_xlabel(
        (
            f"Change in {nombre_clase} "
            "(percentage points)"
        ),
        fontweight="normal"
    )

    eje.set_ylabel(
        UNIDADES_PARAMETROS[parametro],
        fontweight="normal"
    )

    texto_estadistico = crear_texto_estadistico(
        estadisticos
    )

    eje.text(
        0.03,
        0.97,
        texto_estadistico,
        transform=eje.transAxes,
        ha="left",
        va="top",
        fontsize=10.2,
        fontweight="normal",
        bbox={
            "boxstyle": "round,pad=0.35",
            "facecolor": "white",
            "edgecolor": "#BDBDBD",
            "alpha": 0.90,
            "linewidth": 0.7
        },
        zorder=6
    )

    eje.grid(
        axis="both",
        linestyle="--",
        linewidth=0.5,
        alpha=0.30
    )

    eje.set_axisbelow(True)

    eje.spines["top"].set_visible(False)
    eje.spines["right"].set_visible(False)

    eje.tick_params(
        axis="both",
        direction="out"
    )

    eje.margins(
        x=0.10,
        y=0.15
    )

    if mostrar_leyenda:

        eje.legend(
            frameon=False
        )

    return estadisticos, datos


# ============================================================
# 17. FIGURA COMBINADA
# ============================================================

def crear_figura_combinada(tabla):

    figura, ejes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(15.5, 9.5)
    )

    ejes = ejes.flatten()

    resultados = []

    for indice, relacion in enumerate(
        RELACIONES_RELEVANTES
    ):

        eje = ejes[indice]

        estadisticos, datos_relacion = (
            graficar_relacion(
                eje=eje,
                datos=tabla,
                clase=relacion["clase"],
                parametro=relacion["parametro"],
                titulo=relacion["titulo"]
            )
        )

        resultados.append({
            "clase":
                NOMBRES_CLASES[
                    relacion["clase"]
                ],
            "parametro":
                NOMBRES_PARAMETROS[
                    relacion["parametro"]
                ],
            "n":
                estadisticos["n"],
            "pearson_r":
                estadisticos["pearson_r"],
            "pearson_p":
                estadisticos["pearson_p"],
            "spearman_rho":
                estadisticos["spearman_rho"],
            "spearman_p":
                estadisticos["spearman_p"],
            "parcial_r_periodo":
                estadisticos["parcial_r"],
            "parcial_p":
                estadisticos["parcial_p"],
            "pendiente_ajustada":
                estadisticos[
                    "pendiente_ajustada"
                ],
            "pendiente_p":
                estadisticos["pendiente_p"],
            "pendiente_ic95_inferior":
                estadisticos[
                    "pendiente_ic95_inferior"
                ],
            "pendiente_ic95_superior":
                estadisticos[
                    "pendiente_ic95_superior"
                ]
        })

    # Se elimina el sexto panel vacío.
    figura.delaxes(
        ejes[5]
    )

    elementos_leyenda = [
        Line2D(
            [0],
            [0],
            marker=MARCADORES_PERIODOS[periodo],
            color=COLORES_PERIODOS[periodo],
            markerfacecolor=COLORES_PERIODOS[periodo],
            markeredgecolor="white",
            linewidth=1.5,
            markersize=7,
            label=periodo
        )
        for periodo in [
            "2013-2015",
            "2015-2017"
        ]
    ]

    figura.legend(
        handles=elementos_leyenda,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=2,
        frameon=False,
        columnspacing=2.0,
        handletextpad=0.7
    )

    figura.suptitle(
        (
            "Relationships between land-cover changes "
            "and water quality"
        ),
        fontsize=16,
        fontweight="normal",
        y=0.99
    )

    figura.text(
        0.5,
        0.055,
        (
            "The lines represent the model "
            "adjusted for time period"
        ),
        ha="center",
        va="center",
        fontsize=11,
        fontweight="normal",
        color="#444444"
    )

    figura.tight_layout(
        rect=[0, 0.09, 1, 0.96],
        h_pad=2.2,
        w_pad=1.5
    )

    ruta_png = (
        CARPETA_SALIDA
        / "Dispersion_relaciones_relevantes.png"
    )

    ruta_pdf = (
        CARPETA_SALIDA
        / "Dispersion_relaciones_relevantes.pdf"
    )

    ruta_svg = (
        CARPETA_SALIDA
        / "Dispersion_relaciones_relevantes.svg"
    )

    figura.savefig(
        ruta_png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_svg,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    return (
        pd.DataFrame(resultados),
        ruta_png,
        ruta_pdf,
        ruta_svg
    )


# ============================================================
# 18. FIGURAS INDIVIDUALES
# ============================================================

def nombre_archivo_seguro(texto):

    texto = (
        texto
        .replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ñ", "n")
    )

    texto = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")


def crear_figuras_individuales(tabla):

    rutas = []

    for relacion in RELACIONES_RELEVANTES:

        figura, eje = plt.subplots(
            figsize=(7.5, 6)
        )

        estadisticos, datos_relacion = (
            graficar_relacion(
                eje=eje,
                datos=tabla,
                clase=relacion["clase"],
                parametro=relacion["parametro"],
                titulo=relacion["titulo"]
            )
        )

        elementos_leyenda = [
            Line2D(
                [0],
                [0],
                marker=MARCADORES_PERIODOS[periodo],
                color=COLORES_PERIODOS[periodo],
                markerfacecolor=
                    COLORES_PERIODOS[periodo],
                markeredgecolor="white",
                linewidth=1.5,
                markersize=7,
                label=periodo
            )
            for periodo in [
                "2013-2015",
                "2015-2017"
            ]
        ]

        eje.legend(
            handles=elementos_leyenda,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.14),
            ncol=2,
            frameon=False
        )

        figura.tight_layout()

        nombre_base = nombre_archivo_seguro(
            relacion["titulo"]
        )

        ruta_png = (
            CARPETA_SALIDA
            / f"Dispersion_{nombre_base}.png"
        )

        ruta_pdf = (
            CARPETA_SALIDA
            / f"Dispersion_{nombre_base}.pdf"
        )

        figura.savefig(
            ruta_png,
            dpi=600,
            bbox_inches="tight",
            facecolor="white"
        )

        figura.savefig(
            ruta_pdf,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()

        rutas.extend([
            ruta_png,
            ruta_pdf
        ])

    return rutas


# ============================================================
# 19. EJECUCIÓN
# ============================================================

print(
    "\nGenerating combined figure..."
)

(
    resumen_relaciones,
    ruta_png,
    ruta_pdf,
    ruta_svg
) = crear_figura_combinada(
    tabla_exacta
)

print(
    "\nGenerating individual figures..."
)

rutas_individuales = (
    crear_figuras_individuales(
        tabla_exacta
    )
)


# ============================================================
# 20. TABLA DE DATOS UTILIZADOS
# ============================================================

datos_relaciones = []

for relacion in RELACIONES_RELEVANTES:

    seleccion = tabla_exacta[
        (
            tabla_exacta["clase"]
            == relacion["clase"]
        )
        &
        (
            tabla_exacta["parametro"]
            == relacion["parametro"]
        )
    ].copy()

    seleccion["relacion"] = (
        relacion["titulo"]
    )

    datos_relaciones.append(
        seleccion
    )

datos_relaciones = pd.concat(
    datos_relaciones,
    ignore_index=True
)


# ============================================================

# English headers used in every exported table.
OUTPUT_COLUMNS_EN = {'anio': 'year', 'punto': 'point', 'zona': 'zone', 'parametro': 'parameter', 'estimacion': 'estimate', 'archivo_origen': 'source_file', 'observacion': 'observation', 'numero_estimaciones': 'number_of_estimates', 'media_geometrica': 'geometric_mean', 'media_aritmetica': 'arithmetic_mean', 'mediana': 'median', 'minimo': 'minimum', 'maximo': 'maximum', 'desviacion_estandar': 'standard_deviation', 'error_inferior': 'lower_error', 'error_superior': 'upper_error', 'periodo_calidad': 'water_quality_period', 'anio_inicial_calidad': 'water_quality_initial_year', 'anio_final_calidad': 'water_quality_final_year', 'valor_inicial': 'initial_value', 'valor_final': 'final_value', 'cambio_log': 'log_change', 'cambio_porcentual': 'percent_change', 'periodo_suelo': 'land_use_period', 'anio_inicial_suelo': 'land_use_initial_year', 'anio_final_suelo': 'land_use_final_year', 'clase': 'class', 'nombre_clase': 'class_name', 'area_total_zona_ha': 'total_zone_area_ha', 'area_inicial_clase_ha': 'initial_class_area_ha', 'area_final_clase_ha': 'final_class_area_ha', 'persistencia_ha': 'persistence_ha', 'perdida_ha': 'loss_ha', 'ganancia_ha': 'gain_ha', 'cambio_neto_ha': 'net_change_ha', 'cambio_total_ha': 'total_change_ha', 'ganancia_pp': 'gain_pp', 'perdida_pp': 'loss_pp', 'recambio_pp': 'turnover_pp', 'escenario': 'scenario', 'parcial_r_periodo': 'partial_r_period', 'parcial_p': 'partial_p', 'parcial_p_ajustado_BH': 'partial_p_adjusted_BH', 'significativa_BH_0_05': 'significant_BH_0_05', 'area_ha': 'area_ha', 'fuente': 'source', 'superficie_total_ha': 'total_area_ha', 'porcentaje': 'percentage', 'diferencia_ha': 'difference_ha', 'diferencia_absoluta_ha': 'absolute_difference_ha', 'escenario_nombre': 'scenario_name', 'clase_nombre': 'class_name', 'parametro_nombre': 'parameter_name', 'parcial_p_BH_nuevo': 'partial_p_BH'}

def for_export(table):
    return table.rename(columns=OUTPUT_COLUMNS_EN)

# 21. EXPORTACIÓN A EXCEL
# ============================================================

ruta_excel = (
    CARPETA_SALIDA
    / "Scatterplot_data.xlsx"
)

with pd.ExcelWriter(
    ruta_excel,
    engine="openpyxl"
) as writer:

    for_export(datos_relaciones).to_excel(
        writer,
        sheet_name="Plotted_data",
        index=False
    )

    for_export(resumen_relaciones).to_excel(
        writer,
        sheet_name="Statistics",
        index=False
    )


# ============================================================
# 22. PRESENTACIÓN DE LOS RESULTADOS
# ============================================================

print(
    "\nStatistics included in the figures:"
)

display(
    resumen_relaciones.round(4)
)


# ============================================================
# 23. MENSAJE FINAL
# ============================================================

print("\n" + "=" * 68)

print(
    "PROCESS COMPLETED SUCCESSFULLY"
)

print("=" * 68)

print(
    "\nCombined figure:"
)

print(ruta_png)
print(ruta_pdf)
print(ruta_svg)

print(
    "\nLibro Excel:"
)

print(ruta_excel)

print(
    "\nAll results were saved in:"
)

print(CARPETA_SALIDA)

In [ ]:
# ============================================================
# GRÁFICAS DE DISPERSIÓN
# RELACIONES ENTRE CAMBIO DE COBERTURA Y CALIDAD DEL AGUA
# ============================================================
#
# Exact temporal scenario:
# 2013-2015 y 2015-2017
#
# Cada punto representa:
# zona de escurrimiento × periodo
#
# Eje horizontal:
# Cambio neto de la cobertura en puntos porcentuales.
#
# Eje vertical:
# LRC = Log-ratio change
# 100 × ln(valor final / valor inicial)
#
# Las líneas representan regresiones ajustadas por periodo,
# con una pendiente común y diferente intercepto temporal.
# ============================================================


# ============================================================
# 1. INSTALACIÓN DE PAQUETES
# ============================================================



# ============================================================
# 2. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

from pathlib import Path

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import stats
from matplotlib.lines import Line2D
from IPython.display import display


# ============================================================
# 3. CONEXIÓN CON GOOGLE DRIVE
# ============================================================



# ============================================================
# 4. CARPETAS
# ============================================================

CARPETA_PRINCIPAL = REPO_ROOT / "results" / "correlation"

ARCHIVO_RESULTADOS = (
    CARPETA_PRINCIPAL / "Unified_land_use_water_quality_results.xlsx"
)

CARPETA_SALIDA = (
    CARPETA_PRINCIPAL / "Scatterplot_figures"
)

CARPETA_SALIDA.mkdir(
    parents=True,
    exist_ok=True
)

print("Input file:")
print(ARCHIVO_RESULTADOS)

print("\nOutput folder:")
print(CARPETA_SALIDA)


# ============================================================
# 5. NOMBRES DE LAS CLASES
# ============================================================

NOMBRES_CLASES = {
    1: "Water",
    2: "Forest",
    3: "Agricultural land",
    4: "Tropical forests",
    5: "Bare land",
}


# ============================================================
# 6. RELACIONES QUE SE GRAFICARÁN
# ============================================================

RELACIONES_RELEVANTES = [
    {
        "clase": 3,
        "parametro": "COT",
        "titulo": "Agricultural land and TOC"
    },
    {
        "clase": 1,
        "parametro": "COT",
        "titulo": "Water and TOC"
    },
    {
        "clase": 5,
        "parametro": "COT",
        "titulo": "Bare land and TOC"
    },
    {
        "clase": 5,
        "parametro": "SDT",
        "titulo": "Bare land and TDS"
    },
    {
        "clase": 3,
        "parametro": "Cl_a",
        "titulo": "Agricultural land and Chl-a"
    },
]


# ============================================================
# 7. NOMBRES Y ETIQUETAS DE LOS PARÁMETROS
# ============================================================

NOMBRES_PARAMETROS = {
    "COT": "TOC",
    "SDT": "TDS",
    "Cl_a": "Chl-a",
    "Cl-a": "Chl-a",
}

# LRC = Log-ratio change
# Calculado como:
# 100 × ln(WQ_t2 / WQ_t1)
#
# Se omite (%) en el eje porque esta medida no corresponde
# al cambio porcentual convencional.

UNIDADES_PARAMETROS = {
    "COT": "LRC in TOC",
    "SDT": "LRC in TDS",
    "Cl_a": "LRC in Chl-a",
    "Cl-a": "LRC in Chl-a",
}


# ============================================================
# 8. COLORES Y SÍMBOLOS DE LOS PERIODOS
# ============================================================

COLORES_PERIODOS = {
    "2013-2015": "#1F77B4",
    "2015-2017": "#D62728",
}

MARCADORES_PERIODOS = {
    "2013-2015": "o",
    "2015-2017": "s",
}


# ============================================================
# 9. CONFIGURACIÓN GENERAL DE LAS GRÁFICAS
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "axes.linewidth": 0.8,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 120,
    "savefig.dpi": 600,
})


# ============================================================
# 10. VERIFICACIÓN Y LECTURA DEL ARCHIVO
# ============================================================

if not ARCHIVO_RESULTADOS.exists():

    raise FileNotFoundError(
        "No se encontró el archivo de resultados:\n"
        f"{ARCHIVO_RESULTADOS}\n\n"
        "Primero debes ejecutar el código de correlación."
    )

tabla_exacta = pd.read_excel(
    ARCHIVO_RESULTADOS,
    sheet_name="Exact_analysis_table"
)

INPUT_COLUMNS_INTERNAL = {
    'year': 'anio',
    'point': 'punto',
    'zone': 'zona',
    'parameter': 'parametro',
    'estimate': 'estimacion',
    'source_file': 'archivo_origen',
    'observation': 'observacion',
    'number_of_estimates': 'numero_estimaciones',
    'geometric_mean': 'media_geometrica',
    'arithmetic_mean': 'media_aritmetica',
    'median': 'mediana',
    'minimum': 'minimo',
    'maximum': 'maximo',
    'standard_deviation': 'desviacion_estandar',
    'lower_error': 'error_inferior',
    'upper_error': 'error_superior',
    'water_quality_period': 'periodo_calidad',
    'water_quality_initial_year': 'anio_inicial_calidad',
    'water_quality_final_year': 'anio_final_calidad',
    'initial_value': 'valor_inicial',
    'final_value': 'valor_final',
    'log_change': 'cambio_log',
    'percent_change': 'cambio_porcentual',
    'land_use_period': 'periodo_suelo',
    'land_use_initial_year': 'anio_inicial_suelo',
    'land_use_final_year': 'anio_final_suelo',
    'class': 'clase',
    'class_name': 'clase_nombre',
    'total_zone_area_ha': 'area_total_zona_ha',
    'initial_class_area_ha': 'area_inicial_clase_ha',
    'final_class_area_ha': 'area_final_clase_ha',
    'persistence_ha': 'persistencia_ha',
    'loss_ha': 'perdida_ha',
    'gain_ha': 'ganancia_ha',
    'net_change_ha': 'cambio_neto_ha',
    'total_change_ha': 'cambio_total_ha',
    'gain_pp': 'ganancia_pp',
    'loss_pp': 'perdida_pp',
    'turnover_pp': 'recambio_pp',
    'scenario': 'escenario',
    'partial_r_period': 'parcial_r_periodo',
    'partial_p': 'parcial_p',
    'partial_p_adjusted_BH': 'parcial_p_ajustado_BH',
    'significant_BH_0_05': 'significativa_BH_0_05',
    'area_ha': 'area_ha',
    'source': 'fuente',
    'total_area_ha': 'superficie_total_ha',
    'percentage': 'porcentaje',
    'difference_ha': 'diferencia_ha',
    'absolute_difference_ha': 'diferencia_absoluta_ha',
    'scenario_name': 'escenario_nombre',
    'parameter_name': 'parametro_nombre',
    'partial_p_BH': 'parcial_p_BH_nuevo'
}

tabla_exacta = tabla_exacta.rename(
    columns=INPUT_COLUMNS_INTERNAL
)

print(
    "\nRegistros encontrados en la tabla analítica:",
    len(tabla_exacta)
)

print("\nAvailable columns:")

for columna in tabla_exacta.columns:
    print(" -", columna)


# ============================================================
# 11. NORMALIZACIÓN DE COLUMNAS
# ============================================================

columnas_necesarias = [
    "zona",
    "periodo_suelo",
    "clase",
    "parametro",
    "delta_pp",
    "cambio_log"
]

columnas_faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in tabla_exacta.columns
]

if columnas_faltantes:

    raise ValueError(
        "Faltan las siguientes columnas en "
        f"Tabla_analitica_exacta: {columnas_faltantes}"
    )

tabla_exacta["zona"] = (
    tabla_exacta["zona"].astype(int)
)

tabla_exacta["clase"] = (
    tabla_exacta["clase"].astype(int)
)

tabla_exacta["periodo_suelo"] = (
    tabla_exacta["periodo_suelo"].astype(str)
)

tabla_exacta["parametro"] = (
    tabla_exacta["parametro"]
    .replace({
        "Cl-a": "Cl_a",
        "Cha-a": "Cl_a"
    })
)

tabla_exacta["nombre_clase"] = (
    tabla_exacta["clase"]
    .map(NOMBRES_CLASES)
)

tabla_exacta["etiqueta_zona"] = (
    "Z" + tabla_exacta["zona"].astype(str)
)


# ============================================================
# 12. CORRELACIÓN PARCIAL CONTROLANDO EL PERIODO
# ============================================================

def correlacion_parcial_periodo(datos):

    residuo_x = (
        datos["delta_pp"]
        -
        datos.groupby("periodo_suelo")[
            "delta_pp"
        ].transform("mean")
    )

    residuo_y = (
        datos["cambio_log"]
        -
        datos.groupby("periodo_suelo")[
            "cambio_log"
        ].transform("mean")
    )

    resultado = stats.pearsonr(
        residuo_x,
        residuo_y
    )

    return (
        resultado.statistic,
        resultado.pvalue
    )


# ============================================================
# 13. REGRESIÓN AJUSTADA POR PERIODO
# ============================================================

def regresion_ajustada_periodo(datos):

    periodos = sorted(
        datos["periodo_suelo"].unique()
    )

    if len(periodos) != 2:

        raise ValueError(
            "Se esperaban exactamente dos periodos."
        )

    indicador_periodo = (
        datos["periodo_suelo"] == periodos[1]
    ).astype(float).to_numpy()

    x = datos["delta_pp"].to_numpy()
    y = datos["cambio_log"].to_numpy()

    matriz_x = np.column_stack([
        np.ones(len(datos)),
        x,
        indicador_periodo
    ])

    coeficientes = np.linalg.lstsq(
        matriz_x,
        y,
        rcond=None
    )[0]

    intercepto = coeficientes[0]
    pendiente = coeficientes[1]
    efecto_periodo = coeficientes[2]

    predicciones = matriz_x @ coeficientes
    residuos = y - predicciones

    numero_parametros = matriz_x.shape[1]
    grados_libertad = len(y) - numero_parametros

    suma_cuadrados_residuos = (
        residuos @ residuos
    )

    varianza_residual = (
        suma_cuadrados_residuos
        / grados_libertad
    )

    covarianza = (
        varianza_residual
        *
        np.linalg.inv(
            matriz_x.T @ matriz_x
        )
    )

    error_estandar_pendiente = (
        np.sqrt(
            np.diag(covarianza)
        )[1]
    )

    estadistico_t = (
        pendiente
        / error_estandar_pendiente
    )

    valor_p = (
        2
        * stats.t.sf(
            abs(estadistico_t),
            grados_libertad
        )
    )

    valor_critico = stats.t.ppf(
        0.975,
        grados_libertad
    )

    intervalo_inferior = (
        pendiente
        -
        valor_critico
        * error_estandar_pendiente
    )

    intervalo_superior = (
        pendiente
        +
        valor_critico
        * error_estandar_pendiente
    )

    return {
        "periodos": periodos,
        "intercepto": intercepto,
        "pendiente": pendiente,
        "efecto_periodo": efecto_periodo,
        "error_estandar": error_estandar_pendiente,
        "estadistico_t": estadistico_t,
        "valor_p": valor_p,
        "ic95_inferior": intervalo_inferior,
        "ic95_superior": intervalo_superior
    }


# ============================================================
# 14. CÁLCULO DE ESTADÍSTICOS
# ============================================================

def calcular_estadisticos(datos):

    datos = datos.dropna(
        subset=[
            "delta_pp",
            "cambio_log"
        ]
    ).copy()

    resultado_pearson = stats.pearsonr(
        datos["delta_pp"],
        datos["cambio_log"]
    )

    resultado_spearman = stats.spearmanr(
        datos["delta_pp"],
        datos["cambio_log"]
    )

    parcial_r, parcial_p = (
        correlacion_parcial_periodo(
            datos
        )
    )

    regresion = regresion_ajustada_periodo(
        datos
    )

    return {
        "n": len(datos),
        "pearson_r":
            resultado_pearson.statistic,
        "pearson_p":
            resultado_pearson.pvalue,
        "spearman_rho":
            resultado_spearman.statistic,
        "spearman_p":
            resultado_spearman.pvalue,
        "parcial_r":
            parcial_r,
        "parcial_p":
            parcial_p,
        "pendiente_ajustada":
            regresion["pendiente"],
        "pendiente_p":
            regresion["valor_p"],
        "pendiente_ic95_inferior":
            regresion["ic95_inferior"],
        "pendiente_ic95_superior":
            regresion["ic95_superior"],
        "regresion":
            regresion
    }


# ============================================================
# 15. TEXTO ESTADÍSTICO PARA CADA PANEL
# ============================================================

def crear_texto_estadistico(estadisticos):

    texto = (
        f"Pearson: r = "
        f"{estadisticos['pearson_r']:.2f}, "
        f"p = {estadisticos['pearson_p']:.3f}\n"
        f"Spearman: ρ = "
        f"{estadisticos['spearman_rho']:.2f}, "
        f"p = {estadisticos['spearman_p']:.3f}\n"
        f"Partial: r = "
        f"{estadisticos['parcial_r']:.2f}, "
        f"p = {estadisticos['parcial_p']:.3f}"
    )

    return texto


# ============================================================
# 16. FUNCIÓN PARA GRAFICAR UNA RELACIÓN
# ============================================================

def graficar_relacion(
    eje,
    datos,
    clase,
    parametro,
    titulo,
    mostrar_leyenda=False
):

    datos = datos[
        (
            datos["clase"] == clase
        )
        &
        (
            datos["parametro"] == parametro
        )
    ].copy()

    datos = datos.dropna(
        subset=[
            "delta_pp",
            "cambio_log"
        ]
    )

    if len(datos) != 12:

        print(
            f"Advertencia: {titulo} contiene "
            f"{len(datos)} observaciones, no 12."
        )

    estadisticos = calcular_estadisticos(
        datos
    )

    regresion = estadisticos[
        "regresion"
    ]

    periodos = regresion["periodos"]

    # Rango horizontal para las líneas.
    margen_x = (
        datos["delta_pp"].max()
        -
        datos["delta_pp"].min()
    ) * 0.08

    if margen_x == 0:
        margen_x = 0.5

    x_linea = np.linspace(
        datos["delta_pp"].min() - margen_x,
        datos["delta_pp"].max() + margen_x,
        100
    )

    for indice_periodo, periodo in enumerate(
        periodos
    ):

        datos_periodo = datos[
            datos["periodo_suelo"] == periodo
        ].copy()

        color = COLORES_PERIODOS[periodo]
        marcador = MARCADORES_PERIODOS[periodo]

        # Puntos.
        eje.scatter(
            datos_periodo["delta_pp"],
            datos_periodo["cambio_log"],
            s=65,
            marker=marcador,
            facecolor=color,
            edgecolor="white",
            linewidth=0.8,
            alpha=0.95,
            label=periodo,
            zorder=4
        )

        # Identificación de las zonas.
        for _, fila in datos_periodo.iterrows():

            eje.annotate(
                fila["etiqueta_zona"],
                xy=(
                    fila["delta_pp"],
                    fila["cambio_log"]
                ),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=10,
                color="#333333",
                fontweight="normal",
                zorder=5
            )

        # Línea ajustada con pendiente común.
        intercepto_periodo = (
            regresion["intercepto"]
            +
            indice_periodo
            * regresion["efecto_periodo"]
        )

        y_linea = (
            intercepto_periodo
            +
            regresion["pendiente"]
            * x_linea
        )

        eje.plot(
            x_linea,
            y_linea,
            color=color,
            linewidth=1.5,
            linestyle="-",
            alpha=0.85,
            zorder=2
        )

    # Líneas de referencia en cero.
    eje.axhline(
        0,
        color="#666666",
        linewidth=0.7,
        linestyle="--",
        alpha=0.55,
        zorder=1
    )

    eje.axvline(
        0,
        color="#666666",
        linewidth=0.7,
        linestyle="--",
        alpha=0.55,
        zorder=1
    )

    nombre_clase = NOMBRES_CLASES[
        clase
    ]

    eje.set_title(
        titulo,
        fontweight="normal",
        pad=8
    )

    eje.set_xlabel(
        (
            f"Change in {nombre_clase} "
            "(percentage points)"
        ),
        fontweight="normal"
    )

    # --------------------------------------------------------
    # NUEVA ETIQUETA DEL EJE Y
    # --------------------------------------------------------
    eje.set_ylabel(
        UNIDADES_PARAMETROS[parametro],
        fontweight="normal"
    )

    texto_estadistico = crear_texto_estadistico(
        estadisticos
    )

    eje.text(
        0.03,
        0.97,
        texto_estadistico,
        transform=eje.transAxes,
        ha="left",
        va="top",
        fontsize=10.2,
        fontweight="normal",
        bbox={
            "boxstyle": "round,pad=0.35",
            "facecolor": "white",
            "edgecolor": "#BDBDBD",
            "alpha": 0.90,
            "linewidth": 0.7
        },
        zorder=6
    )

    eje.grid(
        axis="both",
        linestyle="--",
        linewidth=0.5,
        alpha=0.30
    )

    eje.set_axisbelow(True)

    eje.spines["top"].set_visible(False)
    eje.spines["right"].set_visible(False)

    eje.tick_params(
        axis="both",
        direction="out"
    )

    eje.margins(
        x=0.10,
        y=0.15
    )

    if mostrar_leyenda:

        eje.legend(
            frameon=False
        )

    return estadisticos, datos


# ============================================================
# 17. FIGURA COMBINADA
# ============================================================

def crear_figura_combinada(tabla):

    figura, ejes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(15.5, 9.5)
    )

    ejes = ejes.flatten()

    resultados = []

    for indice, relacion in enumerate(
        RELACIONES_RELEVANTES
    ):

        eje = ejes[indice]

        estadisticos, datos_relacion = (
            graficar_relacion(
                eje=eje,
                datos=tabla,
                clase=relacion["clase"],
                parametro=relacion["parametro"],
                titulo=relacion["titulo"]
            )
        )

        resultados.append({
            "clase":
                NOMBRES_CLASES[
                    relacion["clase"]
                ],
            "parametro":
                NOMBRES_PARAMETROS[
                    relacion["parametro"]
                ],
            "n":
                estadisticos["n"],
            "pearson_r":
                estadisticos["pearson_r"],
            "pearson_p":
                estadisticos["pearson_p"],
            "spearman_rho":
                estadisticos["spearman_rho"],
            "spearman_p":
                estadisticos["spearman_p"],
            "parcial_r_periodo":
                estadisticos["parcial_r"],
            "parcial_p":
                estadisticos["parcial_p"],
            "pendiente_ajustada":
                estadisticos[
                    "pendiente_ajustada"
                ],
            "pendiente_p":
                estadisticos["pendiente_p"],
            "pendiente_ic95_inferior":
                estadisticos[
                    "pendiente_ic95_inferior"
                ],
            "pendiente_ic95_superior":
                estadisticos[
                    "pendiente_ic95_superior"
                ]
        })

    # Se elimina el sexto panel vacío.
    figura.delaxes(
        ejes[5]
    )

    elementos_leyenda = [
        Line2D(
            [0],
            [0],
            marker=MARCADORES_PERIODOS[periodo],
            color=COLORES_PERIODOS[periodo],
            markerfacecolor=COLORES_PERIODOS[periodo],
            markeredgecolor="white",
            linewidth=1.5,
            markersize=7,
            label=periodo
        )
        for periodo in [
            "2013-2015",
            "2015-2017"
        ]
    ]

    figura.legend(
        handles=elementos_leyenda,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=2,
        frameon=False,
        columnspacing=2.0,
        handletextpad=0.7
    )

    figura.suptitle(
        (
            "Relationships between land-cover changes "
            "and water quality"
        ),
        fontsize=16,
        fontweight="normal",
        y=0.99
    )

    # Se conserva solamente la explicación de las líneas.
    # La definición de LRC se colocará en el pie de figura
    # del manuscrito.
    figura.text(
        0.5,
        0.055,
        (
            "The lines represent the model "
            "adjusted for time period"
        ),
        ha="center",
        va="center",
        fontsize=11,
        fontweight="normal",
        color="#444444"
    )

    figura.tight_layout(
        rect=[0, 0.09, 1, 0.96],
        h_pad=2.2,
        w_pad=1.5
    )

    ruta_png = (
        CARPETA_SALIDA
        / "Dispersion_relaciones_relevantes.png"
    )

    ruta_pdf = (
        CARPETA_SALIDA
        / "Dispersion_relaciones_relevantes.pdf"
    )

    ruta_svg = (
        CARPETA_SALIDA
        / "Dispersion_relaciones_relevantes.svg"
    )

    figura.savefig(
        ruta_png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    figura.savefig(
        ruta_svg,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    return (
        pd.DataFrame(resultados),
        ruta_png,
        ruta_pdf,
        ruta_svg
    )


# ============================================================
# 18. FIGURAS INDIVIDUALES
# ============================================================

def nombre_archivo_seguro(texto):

    texto = (
        texto
        .replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ñ", "n")
    )

    texto = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")


def crear_figuras_individuales(tabla):

    rutas = []

    for relacion in RELACIONES_RELEVANTES:

        figura, eje = plt.subplots(
            figsize=(7.5, 6)
        )

        estadisticos, datos_relacion = (
            graficar_relacion(
                eje=eje,
                datos=tabla,
                clase=relacion["clase"],
                parametro=relacion["parametro"],
                titulo=relacion["titulo"]
            )
        )

        elementos_leyenda = [
            Line2D(
                [0],
                [0],
                marker=MARCADORES_PERIODOS[periodo],
                color=COLORES_PERIODOS[periodo],
                markerfacecolor=
                    COLORES_PERIODOS[periodo],
                markeredgecolor="white",
                linewidth=1.5,
                markersize=7,
                label=periodo
            )
            for periodo in [
                "2013-2015",
                "2015-2017"
            ]
        ]

        eje.legend(
            handles=elementos_leyenda,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.14),
            ncol=2,
            frameon=False
        )

        figura.tight_layout()

        nombre_base = nombre_archivo_seguro(
            relacion["titulo"]
        )

        ruta_png = (
            CARPETA_SALIDA
            / f"Dispersion_{nombre_base}.png"
        )

        ruta_pdf = (
            CARPETA_SALIDA
            / f"Dispersion_{nombre_base}.pdf"
        )

        figura.savefig(
            ruta_png,
            dpi=600,
            bbox_inches="tight",
            facecolor="white"
        )

        figura.savefig(
            ruta_pdf,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.show()

        rutas.extend([
            ruta_png,
            ruta_pdf
        ])

    return rutas


# ============================================================
# 19. EJECUCIÓN
# ============================================================

print(
    "\nGenerating combined figure..."
)

(
    resumen_relaciones,
    ruta_png,
    ruta_pdf,
    ruta_svg
) = crear_figura_combinada(
    tabla_exacta
)

print(
    "\nGenerating individual figures..."
)

rutas_individuales = (
    crear_figuras_individuales(
        tabla_exacta
    )
)


# ============================================================
# 20. TABLA DE DATOS UTILIZADOS
# ============================================================

datos_relaciones = []

for relacion in RELACIONES_RELEVANTES:

    seleccion = tabla_exacta[
        (
            tabla_exacta["clase"]
            == relacion["clase"]
        )
        &
        (
            tabla_exacta["parametro"]
            == relacion["parametro"]
        )
    ].copy()

    seleccion["relacion"] = (
        relacion["titulo"]
    )

    datos_relaciones.append(
        seleccion
    )

datos_relaciones = pd.concat(
    datos_relaciones,
    ignore_index=True
)


# ============================================================
# 21. EXPORTACIÓN A EXCEL
# ============================================================

# English headers used in every exported table.
OUTPUT_COLUMNS_EN = {
    'anio': 'year',
    'punto': 'point',
    'zona': 'zone',
    'parametro': 'parameter',
    'estimacion': 'estimate',
    'archivo_origen': 'source_file',
    'observacion': 'observation',
    'numero_estimaciones': 'number_of_estimates',
    'media_geometrica': 'geometric_mean',
    'media_aritmetica': 'arithmetic_mean',
    'mediana': 'median',
    'minimo': 'minimum',
    'maximo': 'maximum',
    'desviacion_estandar': 'standard_deviation',
    'error_inferior': 'lower_error',
    'error_superior': 'upper_error',
    'periodo_calidad': 'water_quality_period',
    'anio_inicial_calidad': 'water_quality_initial_year',
    'anio_final_calidad': 'water_quality_final_year',
    'valor_inicial': 'initial_value',
    'valor_final': 'final_value',
    'cambio_log': 'log_change',
    'cambio_porcentual': 'percent_change',
    'periodo_suelo': 'land_use_period',
    'anio_inicial_suelo': 'land_use_initial_year',
    'anio_final_suelo': 'land_use_final_year',
    'clase': 'class',
    'nombre_clase': 'class_name',
    'area_total_zona_ha': 'total_zone_area_ha',
    'area_inicial_clase_ha': 'initial_class_area_ha',
    'area_final_clase_ha': 'final_class_area_ha',
    'persistencia_ha': 'persistence_ha',
    'perdida_ha': 'loss_ha',
    'ganancia_ha': 'gain_ha',
    'cambio_neto_ha': 'net_change_ha',
    'cambio_total_ha': 'total_change_ha',
    'ganancia_pp': 'gain_pp',
    'perdida_pp': 'loss_pp',
    'recambio_pp': 'turnover_pp',
    'escenario': 'scenario',
    'parcial_r_periodo': 'partial_r_period',
    'parcial_p': 'partial_p',
    'parcial_p_ajustado_BH':
        'partial_p_adjusted_BH',
    'significativa_BH_0_05':
        'significant_BH_0_05',
    'area_ha': 'area_ha',
    'fuente': 'source',
    'superficie_total_ha': 'total_area_ha',
    'porcentaje': 'percentage',
    'diferencia_ha': 'difference_ha',
    'diferencia_absoluta_ha':
        'absolute_difference_ha',
    'escenario_nombre': 'scenario_name',
    'clase_nombre': 'class_name',
    'parametro_nombre': 'parameter_name',
    'parcial_p_BH_nuevo': 'partial_p_BH'
}


def for_export(table):
    return table.rename(
        columns=OUTPUT_COLUMNS_EN
    )


ruta_excel = (
    CARPETA_SALIDA
    / "Scatterplot_data.xlsx"
)

with pd.ExcelWriter(
    ruta_excel,
    engine="openpyxl"
) as writer:

    for_export(
        datos_relaciones
    ).to_excel(
        writer,
        sheet_name="Plotted_data",
        index=False
    )

    for_export(
        resumen_relaciones
    ).to_excel(
        writer,
        sheet_name="Statistics",
        index=False
    )


# ============================================================
# 22. PRESENTACIÓN DE LOS RESULTADOS
# ============================================================

print(
    "\nStatistics included in the figures:"
)

display(
    resumen_relaciones.round(4)
)


# ============================================================
# 23. MENSAJE FINAL
# ============================================================

print("\n" + "=" * 68)

print(
    "PROCESS COMPLETED SUCCESSFULLY"
)

print("=" * 68)

print(
    "\nCombined figure:"
)

print(ruta_png)
print(ruta_pdf)
print(ruta_svg)

print(
    "\nLibro Excel:"
)

print(ruta_excel)

print(
    "\nAll results were saved in:"
)

print(CARPETA_SALIDA)